In [1]:
import torch

assert torch.cuda.is_available(), "CUDA is unavailable; check the selected notebook kernel."

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Selected device: {DEVICE}")
print(f"GPU: {GPU_NAME}")

PyTorch: 2.5.1
CUDA available: True
Selected device: cuda
GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
from pathlib import Path
import gzip
import json

import numpy as np


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

REPOSITORY_DATA_ROOT = Path.cwd() / "repro_data" / "Assignment_6_v2"
LEGACY_SESSION6_ROOT = Path(
    r"C:\Users\udisi\Documents\Codex\2026-08-01\now"
    r"\outputs\Assignment_6_v2"
)
SESSION6_ROOT = (
    REPOSITORY_DATA_ROOT
    if REPOSITORY_DATA_ROOT.exists()
    else LEGACY_SESSION6_ROOT
)
print(f"Session 6 data root: {SESSION6_ROOT.resolve()}")

PACKED_DIR = SESSION6_ROOT / "data" / "packed_v2"
TOKENIZER_PATH = SESSION6_ROOT / "artifacts" / "tokenizer_v2" / "tokenizer.json"
SELECTION_PATH = SESSION6_ROOT / "artifacts" / "tokenizer_v2" / "selection.json"
PACKING_REPORT_PATH = PACKED_DIR / "packing_report.json"
SEQUENCES_PATH = PACKED_DIR / "sequences.jsonl.gz"

required_paths = [
    TOKENIZER_PATH,
    SELECTION_PATH,
    PACKING_REPORT_PATH,
    SEQUENCES_PATH,
    PACKED_DIR / "input_ids.uint16.bin",
    PACKED_DIR / "loss_mask.uint8.bin",
    PACKED_DIR / "segment_ids.int16.bin",
    PACKED_DIR / "position_ids.uint16.bin",
]

missing = [path for path in required_paths if not path.exists()]
assert not missing, f"Missing Session 6 files:\n{missing}"


# ---------------------------------------------------------------------
# Metadata and tokenizer
# ---------------------------------------------------------------------

with TOKENIZER_PATH.open("r", encoding="utf-8") as handle:
    tokenizer_payload = json.load(handle)

with SELECTION_PATH.open("r", encoding="utf-8") as handle:
    tokenizer_selection = json.load(handle)

with PACKING_REPORT_PATH.open("r", encoding="utf-8") as handle:
    packing_report = json.load(handle)

special_ids = {
    name: int(token_id)
    for name, token_id in tokenizer_payload["special_token_ids"].items()
}

PAD_ID = special_ids["<pad>"]
BOS_ID = special_ids["<bos>"]
EOS_ID = special_ids["<eos>"]
VOCAB_SIZE = len(tokenizer_payload["tokens"])

assert VOCAB_SIZE == 8192
assert (PAD_ID, BOS_ID, EOS_ID) == (0, 1, 2)


# Map IDs to human-readable token pieces.
id_to_piece = {
    int(piece["token_id"]): piece["display"]
    for piece in tokenizer_payload["tokens"]
}

special_by_id = {
    token_id: token
    for token, token_id in special_ids.items()
}


def token_string(token_id: int) -> str:
    """Return a visible representation of one tokenizer piece."""
    token_id = int(token_id)

    if token_id in special_by_id:
        return special_by_id[token_id]

    piece = id_to_piece[token_id]

    # Make whitespace visible when inspecting alignment.
    return (
        piece
        .replace(" ", "␠")
        .replace("\n", "\\n")
        .replace("\t", "\\t")
        .replace("\r", "\\r")
    )


print("Tokenizer")
print(f"  type       : {tokenizer_selection['candidate_id']}")
print(f"  vocab size : {VOCAB_SIZE:,}")
print(f"  PAD ID     : {PAD_ID}")
print(f"  BOS ID     : {BOS_ID}")
print(f"  EOS ID     : {EOS_ID}")

print("\nPacked corpus")
print(f"  sequences           : {packing_report['sequences']:,}")
print(f"  physical tokens     : {packing_report['physical_tokens']:,}")
print(f"  non-padding tokens  : {packing_report['nonpadding_tokens']:,}")
print(f"  loss-bearing tokens : {packing_report['loss_bearing_tokens']:,}")
print(f"  padding tokens      : {packing_report['padding_tokens']:,}")
print(f"  utilization         : {packing_report['packing_utilization']:.4%}")
print(f"  loss density        : {packing_report['loss_density']:.4%}")


# ---------------------------------------------------------------------
# Find one multi-document packed sequence and one padded sequence
# ---------------------------------------------------------------------

multi_document_row = None
padded_row = None

with gzip.open(SEQUENCES_PATH, "rt", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)

        if multi_document_row is None and len(row["fragments"]) >= 2:
            multi_document_row = row

        if (
            padded_row is None
            and row["padding_tokens"] > 0
            and row["loss_bearing_tokens"] > 0
        ):
            padded_row = row

        if multi_document_row is not None and padded_row is not None:
            break

assert multi_document_row is not None, "No multi-document sequence found."
assert padded_row is not None, "No padded sequence found."


# ---------------------------------------------------------------------
# Memory-map the packed arrays
#
# np.memmap reads only requested pages from disk. It does not copy the
# complete packed corpus into process memory.
# ---------------------------------------------------------------------

all_input_ids = np.memmap(
    PACKED_DIR / "input_ids.uint16.bin",
    mode="r",
    dtype="<u2",
)

all_loss_mask = np.memmap(
    PACKED_DIR / "loss_mask.uint8.bin",
    mode="r",
    dtype="u1",
)

all_segment_ids = np.memmap(
    PACKED_DIR / "segment_ids.int16.bin",
    mode="r",
    dtype="<i2",
)

all_position_ids = np.memmap(
    PACKED_DIR / "position_ids.uint16.bin",
    mode="r",
    dtype="<u2",
)


def load_sequence(row: dict) -> dict[str, np.ndarray]:
    start = int(row["global_token_offset"])
    length = int(row["sequence_length"])
    end = start + length

    return {
        "input_ids": np.asarray(all_input_ids[start:end]).copy(),
        "loss_mask": np.asarray(all_loss_mask[start:end]).copy(),
        "segment_ids": np.asarray(all_segment_ids[start:end]).copy(),
        "position_ids": np.asarray(all_position_ids[start:end]).copy(),
    }


# ---------------------------------------------------------------------
# Inspect the multi-document boundary
# ---------------------------------------------------------------------

multi = load_sequence(multi_document_row)

input_ids = multi["input_ids"]
loss_mask = multi["loss_mask"]
segment_ids = multi["segment_ids"]
position_ids = multi["position_ids"]

# A boundary target occurs at position i when segment[i] differs from
# segment[i - 1]. The prediction pair is therefore:
#
# hidden[i - 1] -> input_ids[i]
boundary_targets = np.flatnonzero(
    (segment_ids[1:] >= 0)
    & (segment_ids[:-1] >= 0)
    & (segment_ids[1:] != segment_ids[:-1])
) + 1

assert len(boundary_targets) >= 1

print("\nMulti-document sequence")
print(f"  sequence index : {multi_document_row['sequence_index']}")
print(f"  shape          : input_ids[{len(input_ids)}]")
print(f"  fragments      : {len(multi_document_row['fragments'])}")
print(f"  boundaries     : {len(boundary_targets)}")

for target_index in boundary_targets[:5]:
    input_index = target_index - 1

    print(
        f"\n  boundary at target index {target_index}: "
        f"{token_string(input_ids[input_index])!r}"
        f" -> "
        f"{token_string(input_ids[target_index])!r}"
    )
    print(
        f"    segments       : "
        f"{segment_ids[input_index]} -> {segment_ids[target_index]}"
    )
    print(
        f"    position IDs   : "
        f"{position_ids[input_index]} -> {position_ids[target_index]}"
    )
    print(
        f"    target mask    : {int(loss_mask[target_index])}"
    )


# ---------------------------------------------------------------------
# Reconstruct the exact shifted mask used by the loss harness
# ---------------------------------------------------------------------

shifted_origin_mask = loss_mask[1:].astype(bool)

same_document = (
    (segment_ids[:-1] >= 0)
    & (segment_ids[1:] >= 0)
    & (segment_ids[:-1] == segment_ids[1:])
)

effective_shifted_mask = shifted_origin_mask & same_document

print("\nShifted token accounting")
print(f"  all shifted positions      : {len(input_ids) - 1:,}")
print(f"  origin-approved targets    : {shifted_origin_mask.sum():,}")
print(f"  same-document targets      : {same_document.sum():,}")
print(f"  final contributing targets : {effective_shifted_mask.sum():,}")

# Every cross-document target must be excluded.
assert not effective_shifted_mask[boundary_targets - 1].any()


# ---------------------------------------------------------------------
# Inspect padding invariants independently
# ---------------------------------------------------------------------

padded = load_sequence(padded_row)

pad_positions = padded["segment_ids"] == -1

assert pad_positions.any()
assert np.all(padded["input_ids"][pad_positions] == PAD_ID)
assert np.all(padded["loss_mask"][pad_positions] == 0)

print("\nPadded sequence")
print(f"  sequence index       : {padded_row['sequence_index']}")
print(f"  sequence length      : {padded_row['sequence_length']:,}")
print(f"  non-padding tokens   : {(~pad_positions).sum():,}")
print(f"  padding tokens       : {pad_positions.sum():,}")
print(f"  masked padding count : {padded['loss_mask'][pad_positions].sum():,}")

print("\nSTEP 0 PASSED")
print("Session 6 data preserves EOS, padding, loss, and document-boundary metadata.")

Session 6 data root: C:\Users\udisi\Documents\Codex\2026-08-23\one-notebook-one-loss-harness-and\submission\Week 9\repro_data\Assignment_6_v2
Tokenizer
  type       : standard_bpe_byte_fallback
  vocab size : 8,192
  PAD ID     : 0
  BOS ID     : 1
  EOS ID     : 2

Packed corpus
  sequences           : 39,957
  physical tokens     : 11,378,944
  non-padding tokens  : 11,152,347
  loss-bearing tokens : 10,000,000
  padding tokens      : 226,597
  utilization         : 98.0086%
  loss density        : 87.8816%

Multi-document sequence
  sequence index : 42
  shape          : input_ids[256]
  fragments      : 2
  boundaries     : 1

  boundary at target index 153: '<eos>' -> '{"'
    segments       : 0 -> 1
    position IDs   : 152 -> 0
    target mask    : 0

Shifted token accounting
  all shifted positions      : 255
  origin-approved targets    : 180
  same-document targets      : 251
  final contributing targets : 180

Padded sequence
  sequence index       : 42
  sequence length    

In [3]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------
# Model components
# ---------------------------------------------------------------------

class SegmentCausalSelfAttention(nn.Module):
    """
    Causal self-attention that also prevents attention across documents.

    Input:
        hidden      [B, T, D]
        segment_ids [B, T]

    Output:
        attended    [B, T, D]
    """

    def __init__(self, hidden_size: int, num_heads: int) -> None:
        super().__init__()

        if hidden_size % num_heads != 0:
            raise ValueError("hidden_size must be divisible by num_heads")

        self.num_heads = num_heads
        self.head_size = hidden_size // num_heads

        # One projection produces Q, K, and V together.
        self.qkv_projection = nn.Linear(
            hidden_size,
            3 * hidden_size,
            bias=False,
        )

        self.output_projection = nn.Linear(
            hidden_size,
            hidden_size,
            bias=False,
        )

    def forward(
        self,
        hidden: torch.Tensor,
        segment_ids: torch.Tensor,
    ) -> torch.Tensor:
        B, T, D = hidden.shape

        # [B, T, D] -> [B, T, 3D]
        qkv = self.qkv_projection(hidden)

        # Each tensor is [B, T, D].
        query, key, value = qkv.chunk(3, dim=-1)

        # Split D into H attention heads:
        # [B, T, D] -> [B, H, T, D/H]
        query = query.view(
            B, T, self.num_heads, self.head_size
        ).transpose(1, 2)

        key = key.view(
            B, T, self.num_heads, self.head_size
        ).transpose(1, 2)

        value = value.view(
            B, T, self.num_heads, self.head_size
        ).transpose(1, 2)

        # [B, H, T, D/H] @ [B, H, D/H, T]
        # -> [B, H, T, T]
        scores = query @ key.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_size)

        valid_token = segment_ids >= 0

        # [B, T, T]: query and key must belong to the same document.
        same_segment = (
            segment_ids[:, :, None]
            == segment_ids[:, None, :]
        )

        # [T, T]: position t may only attend to positions <= t.
        causal = torch.ones(
            (T, T),
            dtype=torch.bool,
            device=hidden.device,
        ).tril()

        allowed = (
            same_segment
            & valid_token[:, :, None]
            & valid_token[:, None, :]
            & causal[None, :, :]
        )

        # A fully masked softmax row would be undefined. Give padding
        # queries a self-only row, then zero their output afterward.
        identity = torch.eye(
            T,
            dtype=torch.bool,
            device=hidden.device,
        )[None, :, :]

        allowed = allowed | (
            (~valid_token)[:, :, None] & identity
        )

        scores = scores.masked_fill(
            ~allowed[:, None, :, :],
            torch.finfo(scores.dtype).min,
        )

        attention_weights = torch.softmax(scores, dim=-1)

        # [B, H, T, T] @ [B, H, T, D/H]
        # -> [B, H, T, D/H]
        attended = attention_weights @ value

        # Reassemble the attention heads:
        # [B, H, T, D/H] -> [B, T, D]
        attended = (
            attended
            .transpose(1, 2)
            .contiguous()
            .view(B, T, D)
        )

        # Padding positions must not carry information forward.
        attended = attended * valid_token[:, :, None]

        return self.output_projection(attended)


class DecoderBlock(nn.Module):
    """Pre-normalized transformer decoder block."""

    def __init__(
        self,
        hidden_size: int,
        num_heads: int,
        feed_forward_size: int,
    ) -> None:
        super().__init__()

        self.attention_norm = nn.LayerNorm(hidden_size)
        self.attention = SegmentCausalSelfAttention(
            hidden_size,
            num_heads,
        )

        self.ffn_norm = nn.LayerNorm(hidden_size)
        self.ffn_in = nn.Linear(
            hidden_size,
            feed_forward_size,
        )
        self.ffn_out = nn.Linear(
            feed_forward_size,
            hidden_size,
        )

    def forward(
        self,
        hidden: torch.Tensor,
        segment_ids: torch.Tensor,
    ) -> torch.Tensor:
        # Residual path around attention.
        hidden = hidden + self.attention(
            self.attention_norm(hidden),
            segment_ids,
        )

        # Residual path around the feed-forward network.
        hidden = hidden + self.ffn_out(
            F.gelu(
                self.ffn_in(
                    self.ffn_norm(hidden)
                )
            )
        )

        return hidden


class ObservableDecoder(nn.Module):
    """Small decoder trunk that returns hidden states, not logits."""

    def __init__(
        self,
        vocab_size: int,
        maximum_length: int,
        hidden_size: int,
        num_heads: int,
        feed_forward_size: int,
        num_layers: int,
    ) -> None:
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size,
        )

        self.position_embedding = nn.Embedding(
            maximum_length,
            hidden_size,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                hidden_size=hidden_size,
                num_heads=num_heads,
                feed_forward_size=feed_forward_size,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_size)

        self.apply(self._initialize)

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward_from_embeddings(
        self,
        embeddings: torch.Tensor,
        segment_ids: torch.Tensor,
    ) -> torch.Tensor:
        hidden = embeddings

        for block in self.blocks:
            hidden = block(hidden, segment_ids)

        return self.final_norm(hidden)


# ---------------------------------------------------------------------
# Session 6 model configuration
# ---------------------------------------------------------------------

V = 8192
D = 128
MAX_T = 512
NUM_HEADS = 4
FFN_SIZE = 512
NUM_LAYERS = 2

assert V == VOCAB_SIZE

torch.manual_seed(6202026)
torch.cuda.manual_seed_all(6202026)

model = ObservableDecoder(
    vocab_size=V,
    maximum_length=MAX_T,
    hidden_size=D,
    num_heads=NUM_HEADS,
    feed_forward_size=FFN_SIZE,
    num_layers=NUM_LAYERS,
).to(DEVICE)

# The output head maps each D-dimensional hidden state to V logits.
output_head = nn.Linear(
    in_features=D,
    out_features=V,
    bias=False,
).to(DEVICE)

# Tie the output projection to the input embedding table.
# Both modules now reference the same Parameter object.
output_head.weight = model.token_embedding.weight

assert (
    output_head.weight.data_ptr()
    == model.token_embedding.weight.data_ptr()
)


# ---------------------------------------------------------------------
# Load a real batch containing two equally sized packed sequences
# ---------------------------------------------------------------------

batch_rows = []
selected_length = None

with gzip.open(SEQUENCES_PATH, "rt", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)

        if row["loss_bearing_tokens"] == 0:
            continue

        if selected_length is None:
            selected_length = int(row["sequence_length"])

        if int(row["sequence_length"]) == selected_length:
            batch_rows.append(row)

        if len(batch_rows) == 2:
            break

assert len(batch_rows) == 2

batch_arrays = [
    load_sequence(row)
    for row in batch_rows
]

tokens = torch.as_tensor(
    np.stack([row["input_ids"] for row in batch_arrays]),
    dtype=torch.long,
    device=DEVICE,
)

segment_ids = torch.as_tensor(
    np.stack([row["segment_ids"] for row in batch_arrays]),
    dtype=torch.long,
    device=DEVICE,
)

position_ids = torch.as_tensor(
    np.stack([row["position_ids"] for row in batch_arrays]),
    dtype=torch.long,
    device=DEVICE,
)

loss_mask = torch.as_tensor(
    np.stack([row["loss_mask"] for row in batch_arrays]),
    dtype=torch.bool,
    device=DEVICE,
)


# ---------------------------------------------------------------------
# Observable forward pass
# ---------------------------------------------------------------------

model.eval()
output_head.eval()

with torch.no_grad():
    # Embedding lookup: [B, T] -> [B, T, D]
    token_embeddings = model.token_embedding(tokens)
    position_embeddings = model.position_embedding(position_ids)
    embeddings = token_embeddings + position_embeddings

    # Decoder trunk preserves [B, T, D].
    hidden = model.forward_from_embeddings(
        embeddings,
        segment_ids,
    )

    # Vocabulary projection: [B, T, D] -> [B, T, V]
    logits = output_head(hidden)

    # Hidden state at t predicts the token at t+1.
    prediction_logits = logits[:, :-1, :].contiguous()
    targets = tokens[:, 1:].contiguous()

    shifted_origin_mask = loss_mask[:, 1:].contiguous()

    same_segment_mask = (
        (segment_ids[:, :-1] >= 0)
        & (segment_ids[:, 1:] >= 0)
        & (
            segment_ids[:, :-1]
            == segment_ids[:, 1:]
        )
    )

    effective_loss_mask = (
        shifted_origin_mask
        & same_segment_mask
    )

    # These are the exact shapes expected by cross_entropy.
    flat_prediction_logits = prediction_logits.reshape(-1, V)
    flat_targets = targets.reshape(-1)


# ---------------------------------------------------------------------
# Print every loss-harness tensor shape
# ---------------------------------------------------------------------

B, T = tokens.shape

print(
    f"B={B}: sequences per batch | "
    f"T={T}: token positions per sequence | "
    f"D={D}: hidden features per token | "
    f"V={V}: vocabulary scores per position"
)

print()


def show_shape(
    name: str,
    tensor: torch.Tensor,
    axes: str,
    meaning: str,
) -> None:
    print(
        f"{name:<25} "
        f"{str(tuple(tensor.shape)):<18} "
        f"{axes:<15} "
        f"{meaning}"
    )


show_shape(
    "tokens",
    tokens,
    "[B, T]",
    "integer token IDs",
)

show_shape(
    "segment_ids",
    segment_ids,
    "[B, T]",
    "document identity; -1 means padding",
)

show_shape(
    "position_ids",
    position_ids,
    "[B, T]",
    "position within each document",
)

show_shape(
    "loss_mask",
    loss_mask,
    "[B, T]",
    "target eligibility before shifting",
)

show_shape(
    "token_embeddings",
    token_embeddings,
    "[B, T, D]",
    "learned vector for each token",
)

show_shape(
    "position_embeddings",
    position_embeddings,
    "[B, T, D]",
    "learned vector for each position",
)

show_shape(
    "embeddings",
    embeddings,
    "[B, T, D]",
    "token plus position representation",
)

show_shape(
    "hidden",
    hidden,
    "[B, T, D]",
    "contextual representation from decoder",
)

show_shape(
    "output_head.weight",
    output_head.weight,
    "[V, D]",
    "one learned scoring row per vocabulary token",
)

show_shape(
    "logits",
    logits,
    "[B, T, V]",
    "unnormalized vocabulary scores",
)

show_shape(
    "prediction_logits",
    prediction_logits,
    "[B, T-1, V]",
    "positions that have a next-token target",
)

show_shape(
    "targets",
    targets,
    "[B, T-1]",
    "next-token labels",
)

show_shape(
    "shifted_origin_mask",
    shifted_origin_mask,
    "[B, T-1]",
    "target-aligned Session 6 mask",
)

show_shape(
    "same_segment_mask",
    same_segment_mask,
    "[B, T-1]",
    "rejects cross-document transitions",
)

show_shape(
    "effective_loss_mask",
    effective_loss_mask,
    "[B, T-1]",
    "final positions allowed into the loss",
)

show_shape(
    "flat_prediction_logits",
    flat_prediction_logits,
    "[B(T-1), V]",
    "2D input expected by cross_entropy",
)

show_shape(
    "flat_targets",
    flat_targets,
    "[B(T-1)]",
    "1D labels expected by cross_entropy",
)


# ---------------------------------------------------------------------
# Mechanical correctness assertions
# ---------------------------------------------------------------------

assert tokens.shape == (B, T)
assert embeddings.shape == (B, T, D)
assert hidden.shape == (B, T, D)
assert logits.shape == (B, T, V)
assert prediction_logits.shape == (B, T - 1, V)
assert targets.shape == (B, T - 1)
assert effective_loss_mask.shape == (B, T - 1)
assert flat_prediction_logits.shape == (B * (T - 1), V)
assert flat_targets.shape == (B * (T - 1),)

logits_mib = (
    logits.numel()
    * logits.element_size()
    / 2**20
)

print()
print(f"Materialized logits storage: {logits_mib:.2f} MiB")
print("STEP 1 PASSED")

B=2: sequences per batch | T=256: token positions per sequence | D=128: hidden features per token | V=8192: vocabulary scores per position

tokens                    (2, 256)           [B, T]          integer token IDs
segment_ids               (2, 256)           [B, T]          document identity; -1 means padding
position_ids              (2, 256)           [B, T]          position within each document
loss_mask                 (2, 256)           [B, T]          target eligibility before shifting
token_embeddings          (2, 256, 128)      [B, T, D]       learned vector for each token
position_embeddings       (2, 256, 128)      [B, T, D]       learned vector for each position
embeddings                (2, 256, 128)      [B, T, D]       token plus position representation
hidden                    (2, 256, 128)      [B, T, D]       contextual representation from decoder
output_head.weight        (8192, 128)        [V, D]          one learned scoring row per vocabulary token
logits    

In [4]:
# ---------------------------------------------------------------------
# Load the real multi-document sequence found in Step 0
# ---------------------------------------------------------------------

inspection_arrays = load_sequence(multi_document_row)

inspection_tokens = torch.as_tensor(
    inspection_arrays["input_ids"][None, :],
    dtype=torch.long,
    device=DEVICE,
)

inspection_segments = torch.as_tensor(
    inspection_arrays["segment_ids"][None, :],
    dtype=torch.long,
    device=DEVICE,
)

inspection_loss_mask = torch.as_tensor(
    inspection_arrays["loss_mask"][None, :],
    dtype=torch.bool,
    device=DEVICE,
)


# ---------------------------------------------------------------------
# Apply the exact next-token shift
#
# At position t:
#     hidden[:, t, :] predicts tokens[:, t + 1]
# ---------------------------------------------------------------------

input_token_ids = inspection_tokens[:, :-1].contiguous()
target_token_ids = inspection_tokens[:, 1:].contiguous()

input_segment_ids = inspection_segments[:, :-1].contiguous()
target_segment_ids = inspection_segments[:, 1:].contiguous()

target_origin_mask = inspection_loss_mask[:, 1:].contiguous()

same_document = (
    (input_segment_ids >= 0)
    & (target_segment_ids >= 0)
    & (input_segment_ids == target_segment_ids)
)

target_contributes = target_origin_mask & same_document


# ---------------------------------------------------------------------
# Mechanical shift assertions
# ---------------------------------------------------------------------

assert input_token_ids.shape == target_token_ids.shape
assert torch.equal(
    input_token_ids,
    inspection_tokens[:, :-1],
)
assert torch.equal(
    target_token_ids,
    inspection_tokens[:, 1:],
)

# For every prediction position t, verify directly that the target
# is the original token at t + 1.
for t in range(target_token_ids.shape[1]):
    assert (
        target_token_ids[0, t].item()
        == inspection_tokens[0, t + 1].item()
    )


# ---------------------------------------------------------------------
# Locate the first cross-document prediction pair
# ---------------------------------------------------------------------

boundary_prediction_positions = torch.nonzero(
    (input_segment_ids >= 0)
    & (target_segment_ids >= 0)
    & (input_segment_ids != target_segment_ids),
    as_tuple=False,
)

assert len(boundary_prediction_positions) >= 1

# Columns returned by nonzero are [batch_index, prediction_position].
boundary_t = int(boundary_prediction_positions[0, 1].item())

window_start = max(0, boundary_t - 8)
window_end = min(
    target_token_ids.shape[1],
    boundary_t + 9,
)


# ---------------------------------------------------------------------
# Print actual token strings—not IDs
# ---------------------------------------------------------------------

print(
    "Each row means: logits[:, t, :] must predict the target token at t+1."
)
print()

header = (
    f"{'t':>4} | "
    f"{'INPUT token x[t]':<26} | "
    f"{'TARGET token x[t+1]':<26} | "
    f"{'segments':<9} | "
    f"{'counts?':<7}"
)

print(header)
print("-" * len(header))

for t in range(window_start, window_end):
    input_id = int(input_token_ids[0, t].item())
    target_id = int(target_token_ids[0, t].item())

    input_text = repr(token_string(input_id))
    target_text = repr(token_string(target_id))

    input_segment = int(input_segment_ids[0, t].item())
    target_segment = int(target_segment_ids[0, t].item())

    contributes = bool(target_contributes[0, t].item())

    boundary_marker = "  <-- DOCUMENT BOUNDARY" if t == boundary_t else ""

    print(
        f"{t:>4} | "
        f"{input_text:<26} | "
        f"{target_text:<26} | "
        f"{input_segment}->{target_segment:<6} | "
        f"{str(contributes):<7}"
        f"{boundary_marker}"
    )


# ---------------------------------------------------------------------
# Print the shift as two aligned string rows
# ---------------------------------------------------------------------

display_start = max(0, boundary_t - 5)
display_end = min(
    input_token_ids.shape[1],
    boundary_t + 6,
)

input_strings = [
    token_string(token_id)
    for token_id in input_token_ids[
        0, display_start:display_end
    ].tolist()
]

target_strings = [
    token_string(token_id)
    for token_id in target_token_ids[
        0, display_start:display_end
    ].tolist()
]

print()
print("Aligned string-level shift")
print()

for relative_position, (input_text, target_text) in enumerate(
    zip(input_strings, target_strings),
    start=display_start,
):
    print(
        f"t={relative_position:<3} "
        f"{input_text!r:<26} -> {target_text!r}"
    )


# ---------------------------------------------------------------------
# Explicitly verify the boundary pair
# ---------------------------------------------------------------------

boundary_input_string = token_string(
    input_token_ids[0, boundary_t].item()
)

boundary_target_string = token_string(
    target_token_ids[0, boundary_t].item()
)

print()
print("Boundary verification")
print(
    f"  logits at position {boundary_t} would predict: "
    f"{boundary_input_string!r} -> {boundary_target_string!r}"
)
print(
    f"  origin target mask : "
    f"{bool(target_origin_mask[0, boundary_t].item())}"
)
print(
    f"  same document      : "
    f"{bool(same_document[0, boundary_t].item())}"
)
print(
    f"  contributes to loss: "
    f"{bool(target_contributes[0, boundary_t].item())}"
)

assert boundary_input_string == "<eos>"
assert not same_document[0, boundary_t]
assert not target_contributes[0, boundary_t]

print()
print("STEP 2 PASSED — token strings confirm x[t] predicts x[t+1].")

Each row means: logits[:, t, :] must predict the target token at t+1.

   t | INPUT token x[t]           | TARGET token x[t+1]        | segments  | counts?
------------------------------------------------------------------------------------
 144 | '<0x20>'                   | 'their'                    | 0->0      | True   
 145 | 'their'                    | '<0x20>'                   | 0->0      | True   
 146 | '<0x20>'                   | 'And'                      | 0->0      | True   
 147 | 'And'                      | 'roid'                     | 0->0      | True   
 148 | 'roid'                     | '<0x20>'                   | 0->0      | True   
 149 | '<0x20>'                   | 'device'                   | 0->0      | True   
 150 | 'device'                   | '<0x2E>'                   | 0->0      | True   
 151 | '<0x2E>'                   | '<eos>'                    | 0->0      | True   
 152 | '<eos>'                    | '{"'                       | 0->1      | Fa

In [5]:
# ---------------------------------------------------------------------
# Construct a controlled variable-length batch from real tokens
# ---------------------------------------------------------------------

FULL_LENGTH = 64
SHORT_LENGTH = 47

source_ids = inspection_arrays["input_ids"][:FULL_LENGTH].copy()
source_segments = inspection_arrays["segment_ids"][:FULL_LENGTH].copy()
source_positions = inspection_arrays["position_ids"][:FULL_LENGTH].copy()

# This slice must be entirely inside one real document.
assert len(source_ids) == FULL_LENGTH
assert np.all(source_segments >= 0)
assert np.all(source_segments == source_segments[0])
assert PAD_ID not in source_ids

# Row 0: 64 real tokens.
full_tokens = source_ids.copy()
full_segments = source_segments.copy()
full_positions = source_positions.copy()

# Row 1: 47 real tokens followed by 17 padding positions.
short_tokens = np.full(
    FULL_LENGTH,
    PAD_ID,
    dtype=np.uint16,
)
short_tokens[:SHORT_LENGTH] = source_ids[:SHORT_LENGTH]

short_segments = np.full(
    FULL_LENGTH,
    -1,
    dtype=np.int16,
)
short_segments[:SHORT_LENGTH] = 0

short_positions = np.zeros(
    FULL_LENGTH,
    dtype=np.uint16,
)
short_positions[:SHORT_LENGTH] = np.arange(
    SHORT_LENGTH,
    dtype=np.uint16,
)

padded_tokens = torch.as_tensor(
    np.stack([full_tokens, short_tokens]),
    dtype=torch.long,
    device=DEVICE,
)

padded_segments = torch.as_tensor(
    np.stack([full_segments, short_segments]),
    dtype=torch.long,
    device=DEVICE,
)

padded_positions = torch.as_tensor(
    np.stack([full_positions, short_positions]),
    dtype=torch.long,
    device=DEVICE,
)


# ---------------------------------------------------------------------
# Forward pass
# ---------------------------------------------------------------------

model.eval()
output_head.eval()

with torch.no_grad():
    padded_embeddings = (
        model.token_embedding(padded_tokens)
        + model.position_embedding(padded_positions)
    )

    padded_hidden = model.forward_from_embeddings(
        padded_embeddings,
        padded_segments,
    )

    padded_logits = output_head(padded_hidden)


# ---------------------------------------------------------------------
# Shift logits and targets
# ---------------------------------------------------------------------

shifted_padding_logits = padded_logits[:, :-1, :].contiguous()
shifted_padding_targets = padded_tokens[:, 1:].contiguous()

flat_padding_logits = shifted_padding_logits.reshape(-1, V)
flat_padding_targets = shifted_padding_targets.reshape(-1)


# ---------------------------------------------------------------------
# Method A: incorrect loss—every physical position contributes
# ---------------------------------------------------------------------

per_token_loss = F.cross_entropy(
    flat_padding_logits,
    flat_padding_targets,
    reduction="none",
)

naive_loss = per_token_loss.mean()
naive_count = flat_padding_targets.numel()


# ---------------------------------------------------------------------
# Method B: correct padding-masked loss
#
# A prediction position is masked when its TARGET is <pad>.
# ---------------------------------------------------------------------

active_padding_mask = flat_padding_targets != PAD_ID
active_count = int(active_padding_mask.sum().item())

explicit_masked_loss = per_token_loss[
    active_padding_mask
].mean()

# PyTorch's built-in implementation of the same operation.
ignore_index_loss = F.cross_entropy(
    flat_padding_logits,
    flat_padding_targets,
    ignore_index=PAD_ID,
    reduction="mean",
)

assert torch.allclose(
    explicit_masked_loss,
    ignore_index_loss,
    atol=1e-7,
    rtol=1e-6,
)


# ---------------------------------------------------------------------
# Expected accounting
# ---------------------------------------------------------------------

B_PAD, T_PAD = padded_tokens.shape

expected_naive_count = B_PAD * (T_PAD - 1)
padding_target_count = int(
    (flat_padding_targets == PAD_ID).sum().item()
)
expected_active_count = (
    expected_naive_count - padding_target_count
)

assert naive_count == expected_naive_count
assert active_count == expected_active_count
assert padding_target_count == FULL_LENGTH - SHORT_LENGTH


# ---------------------------------------------------------------------
# Print the exact padding transition as token strings
# ---------------------------------------------------------------------

print(
    f"B={B_PAD}, T={T_PAD}, V={V}"
)
print(
    f"Sequence lengths before padding: "
    f"[{FULL_LENGTH}, {SHORT_LENGTH}]"
)
print()

header = (
    f"{'t':>3} | "
    f"{'INPUT token':<24} | "
    f"{'TARGET token':<24} | "
    f"{'target active?':<14}"
)

print(header)
print("-" * len(header))

# The first padding target is at original token position SHORT_LENGTH,
# so its prediction position is SHORT_LENGTH - 1.
first_padding_prediction_t = SHORT_LENGTH - 1

for t in range(
    first_padding_prediction_t - 3,
    first_padding_prediction_t + 5,
):
    input_id = int(padded_tokens[1, t].item())
    target_id = int(padded_tokens[1, t + 1].item())

    input_text = repr(token_string(input_id))
    target_text = repr(token_string(target_id))

    target_active = target_id != PAD_ID

    marker = (
        "  <-- first masked padding target"
        if t == first_padding_prediction_t
        else ""
    )

    print(
        f"{t:>3} | "
        f"{input_text:<24} | "
        f"{target_text:<24} | "
        f"{str(target_active):<14}"
        f"{marker}"
    )


# ---------------------------------------------------------------------
# Report losses and contributing-token counts
# ---------------------------------------------------------------------

print()
print("Token accounting")
print(
    f"  Before padding mask : "
    f"{naive_count:>3} contributing targets"
)
print(
    f"  Padding targets     : "
    f"{padding_target_count:>3} excluded targets"
)
print(
    f"  After padding mask  : "
    f"{active_count:>3} contributing targets"
)

print()
print("Loss comparison")
print(
    f"  Naive loss, padding included : "
    f"{naive_loss.item():.6f}"
)
print(
    f"  Explicit masked loss         : "
    f"{explicit_masked_loss.item():.6f}"
)
print(
    f"  ignore_index masked loss     : "
    f"{ignore_index_loss.item():.6f}"
)

print()
print(
    f"Count changed: "
    f"{naive_count} -> {active_count} "
    f"({padding_target_count} targets removed)"
)


# Save quantitative results for the final README.
if "part1_metrics" not in globals():
    part1_metrics = {}

part1_metrics["padding"] = {
    "count_before": naive_count,
    "count_after": active_count,
    "padding_targets_removed": padding_target_count,
    "naive_loss": naive_loss.item(),
    "masked_loss": ignore_index_loss.item(),
}

print()
print("STEP 3 PASSED")

B=2, T=64, V=8192
Sequence lengths before padding: [64, 47]

  t | INPUT token              | TARGET token             | target active?
--------------------------------------------------------------------------
 43 | '_c'                     | 'ard'                    | True          
 44 | 'ard'                    | '<0x0A>'                 | True          
 45 | '<0x0A>'                 | '<0x2D>'                 | True          
 46 | '<0x2D>'                 | '<pad>'                  | False           <-- first masked padding target
 47 | '<pad>'                  | '<pad>'                  | False         
 48 | '<pad>'                  | '<pad>'                  | False         
 49 | '<pad>'                  | '<pad>'                  | False         
 50 | '<pad>'                  | '<pad>'                  | False         

Token accounting
  Before padding mask : 126 contributing targets
  Padding targets     :  17 excluded targets
  After padding mask  : 109 contributing tar

In [6]:
# ---------------------------------------------------------------------
# Prepare the real two-document packed sequence
# ---------------------------------------------------------------------

boundary_tokens = inspection_tokens
boundary_segments = inspection_segments

boundary_positions = torch.as_tensor(
    inspection_arrays["position_ids"][None, :],
    dtype=torch.long,
    device=DEVICE,
)

assert boundary_tokens.shape == boundary_segments.shape
assert boundary_tokens.shape == boundary_positions.shape


# ---------------------------------------------------------------------
# Forward pass with document-aware causal attention
# ---------------------------------------------------------------------

model.eval()
output_head.eval()

with torch.no_grad():
    boundary_embeddings = (
        model.token_embedding(boundary_tokens)
        + model.position_embedding(boundary_positions)
    )

    boundary_hidden = model.forward_from_embeddings(
        boundary_embeddings,
        boundary_segments,
    )

    boundary_logits = output_head(boundary_hidden)


# ---------------------------------------------------------------------
# Shift logits and targets
# ---------------------------------------------------------------------

boundary_prediction_logits = (
    boundary_logits[:, :-1, :].contiguous()
)

boundary_targets = (
    boundary_tokens[:, 1:].contiguous()
)

boundary_input_segments = (
    boundary_segments[:, :-1].contiguous()
)

boundary_target_segments = (
    boundary_segments[:, 1:].contiguous()
)


# ---------------------------------------------------------------------
# Compute every shifted token loss without reducing
# ---------------------------------------------------------------------

boundary_token_losses = F.cross_entropy(
    boundary_prediction_logits.reshape(-1, V),
    boundary_targets.reshape(-1),
    reduction="none",
).view_as(boundary_targets)


# ---------------------------------------------------------------------
# Mask A: padding removed, document boundary still included
# ---------------------------------------------------------------------

valid_input = boundary_input_segments >= 0
valid_target = boundary_target_segments >= 0

mask_before_boundary = valid_input & valid_target


# ---------------------------------------------------------------------
# Mask B: padding and cross-document boundaries removed
# ---------------------------------------------------------------------

same_document = (
    boundary_input_segments
    == boundary_target_segments
)

mask_after_boundary = (
    mask_before_boundary
    & same_document
)


# ---------------------------------------------------------------------
# Locate exactly which prediction pair was removed
# ---------------------------------------------------------------------

removed_by_boundary_mask = (
    mask_before_boundary
    & ~mask_after_boundary
)

removed_locations = torch.nonzero(
    removed_by_boundary_mask,
    as_tuple=False,
)

assert removed_locations.shape[0] == 1

removed_batch = int(removed_locations[0, 0].item())
removed_t = int(removed_locations[0, 1].item())

removed_input_id = int(
    boundary_tokens[removed_batch, removed_t].item()
)

removed_target_id = int(
    boundary_targets[removed_batch, removed_t].item()
)

removed_input_string = token_string(removed_input_id)
removed_target_string = token_string(removed_target_id)

assert removed_input_string == "<eos>"


# ---------------------------------------------------------------------
# Mean loss before and after boundary masking
# ---------------------------------------------------------------------

loss_before_boundary_mask = boundary_token_losses[
    mask_before_boundary
].mean()

loss_after_boundary_mask = boundary_token_losses[
    mask_after_boundary
].mean()

boundary_transition_loss = boundary_token_losses[
    removed_batch,
    removed_t,
]

count_before_boundary = int(
    mask_before_boundary.sum().item()
)

count_after_boundary = int(
    mask_after_boundary.sum().item()
)

assert (
    count_before_boundary
    == count_after_boundary + 1
)


# ---------------------------------------------------------------------
# Verify the mean algebra explicitly
#
# L_before = (sum_after + boundary_loss) / (N_after + 1)
# ---------------------------------------------------------------------

loss_sum_after = boundary_token_losses[
    mask_after_boundary
].double().sum()

reconstructed_before = (
    loss_sum_after
    + boundary_transition_loss.double()
) / count_before_boundary

assert torch.allclose(
    loss_before_boundary_mask.double(),
    reconstructed_before,
    atol=1e-6,
    rtol=1e-6,
)


# ---------------------------------------------------------------------
# Probability assigned to the arbitrary boundary target
#
# Since CE = -log(p_target):
#     p_target = exp(-CE)
# ---------------------------------------------------------------------

boundary_target_probability = torch.exp(
    -boundary_transition_loss
)


# ---------------------------------------------------------------------
# Print the packed-document boundary
# ---------------------------------------------------------------------

print("Packed document boundary")
print(
    f"  prediction position : {removed_t}"
)
print(
    f"  token strings       : "
    f"{removed_input_string!r} -> "
    f"{removed_target_string!r}"
)
print(
    f"  segment IDs         : "
    f"{int(boundary_input_segments[removed_batch, removed_t])}"
    f" -> "
    f"{int(boundary_target_segments[removed_batch, removed_t])}"
)
print(
    f"  boundary-token loss : "
    f"{boundary_transition_loss.item():.6f}"
)
print(
    f"  probability assigned: "
    f"{boundary_target_probability.item():.8f}"
)


# ---------------------------------------------------------------------
# Print loss and token-count comparison
# ---------------------------------------------------------------------

print()
print("Boundary-mask comparison")
print(
    f"  Before boundary mask: "
    f"loss={loss_before_boundary_mask.item():.6f}, "
    f"count={count_before_boundary}"
)
print(
    f"  After boundary mask : "
    f"loss={loss_after_boundary_mask.item():.6f}, "
    f"count={count_after_boundary}"
)

loss_change = (
    loss_after_boundary_mask.item()
    - loss_before_boundary_mask.item()
)

print(
    f"  Loss change, after - before: "
    f"{loss_change:+.6f}"
)
print(
    f"  Targets removed: "
    f"{count_before_boundary - count_after_boundary}"
)


# ---------------------------------------------------------------------
# Explain the observed direction numerically
# ---------------------------------------------------------------------

remaining_mean = loss_after_boundary_mask.item()
removed_loss = boundary_transition_loss.item()

print()
if removed_loss > remaining_mean:
    print(
        "The boundary token was harder than the average retained "
        "token, so removing it lowered the mean loss."
    )
elif removed_loss < remaining_mean:
    print(
        "The boundary token was easier than the average retained "
        "token, so removing it raised the mean loss."
    )
else:
    print(
        "The boundary-token loss matched the retained mean, so "
        "removing it did not change the displayed mean."
    )


# ---------------------------------------------------------------------
# Save metrics for the final README
# ---------------------------------------------------------------------

part1_metrics["boundary"] = {
    "token_pair": (
        f"{removed_input_string} -> "
        f"{removed_target_string}"
    ),
    "count_before": count_before_boundary,
    "count_after": count_after_boundary,
    "boundary_token_loss": boundary_transition_loss.item(),
    "loss_before": loss_before_boundary_mask.item(),
    "loss_after": loss_after_boundary_mask.item(),
    "loss_change_after_minus_before": loss_change,
}

print()
print("STEP 4 PASSED")

Packed document boundary
  prediction position : 152
  token strings       : '<eos>' -> '{"'
  segment IDs         : 0 -> 1
  boundary-token loss : 9.396762
  probability assigned: 0.00008299

Boundary-mask comparison
  Before boundary mask: loss=9.055383, count=252
  After boundary mask : loss=9.054023, count=251
  Loss change, after - before: -0.001360
  Targets removed: 1

The boundary token was harder than the average retained token, so removing it lowered the mean loss.

STEP 4 PASSED


In [7]:
# ---------------------------------------------------------------------
# Select only valid next-token training positions
# ---------------------------------------------------------------------

assert prediction_logits.shape[:2] == effective_loss_mask.shape
assert targets.shape == effective_loss_mask.shape

untrained_active_logits = prediction_logits[
    effective_loss_mask
]

untrained_active_targets = targets[
    effective_loss_mask
]

untrained_contributing_count = int(
    untrained_active_targets.numel()
)

assert untrained_contributing_count > 0
assert untrained_active_logits.shape == (
    untrained_contributing_count,
    V,
)


# ---------------------------------------------------------------------
# Compute mean next-token cross-entropy
#
# F.cross_entropy expects raw logits. Do not apply softmax first.
# ---------------------------------------------------------------------

with torch.no_grad():
    untrained_mean_loss = F.cross_entropy(
        untrained_active_logits,
        untrained_active_targets,
        reduction="mean",
    )

    untrained_perplexity = torch.exp(
        untrained_mean_loss
    )


# ---------------------------------------------------------------------
# Exact uniform-distribution reference
# ---------------------------------------------------------------------

uniform_loss = math.log(V)
uniform_perplexity = math.exp(uniform_loss)

# PPL can also be interpreted through the geometric mean probability
# assigned to the correct token:
#
#     geometric_mean_p_target = exp(-mean_loss) = 1 / PPL
observed_geometric_mean_probability = torch.exp(
    -untrained_mean_loss
).item()

uniform_target_probability = 1.0 / V


# ---------------------------------------------------------------------
# Compare the initialized model with the uniform reference
# ---------------------------------------------------------------------

loss_error_nats = (
    untrained_mean_loss.item()
    - uniform_loss
)

perplexity_error = (
    untrained_perplexity.item()
    - V
)

perplexity_ratio_to_vocab = (
    untrained_perplexity.item()
    / V
)

perplexity_relative_error = abs(
    perplexity_ratio_to_vocab - 1.0
)


print("Uniform-reference sanity check")
print(f"  Vocabulary size V             : {V:,}")
print(f"  Expected uniform loss ln(V)   : {uniform_loss:.6f}")
print(f"  Observed untrained mean loss  : {untrained_mean_loss.item():.6f}")
print(f"  Loss difference               : {loss_error_nats:+.6f} nats")

print()
print(f"  Expected uniform perplexity   : {uniform_perplexity:,.2f}")
print(f"  Observed untrained perplexity : {untrained_perplexity.item():,.2f}")
print(f"  Perplexity difference         : {perplexity_error:+,.2f}")
print(f"  PPL / V ratio                 : {perplexity_ratio_to_vocab:.6f}")
print(f"  Relative distance from V      : {perplexity_relative_error:.2%}")

print()
print(f"  Uniform target probability    : {uniform_target_probability:.8f}")
print(
    f"  Observed geometric-mean "
    f"target probability: "
    f"{observed_geometric_mean_probability:.8f}"
)
print(
    f"  Contributing targets          : "
    f"{untrained_contributing_count:,}"
)


# ---------------------------------------------------------------------
# Explicit gate: do not proceed if the model is far from V
#
# A 15% band allows small random-logit dispersion while still catching
# major alignment, reduction, vocabulary, or initialization failures.
# ---------------------------------------------------------------------

MAX_RELATIVE_PPL_ERROR = 0.15

sanity_passed = (
    torch.isfinite(untrained_mean_loss)
    and torch.isfinite(untrained_perplexity)
    and perplexity_relative_error <= MAX_RELATIVE_PPL_ERROR
)

if not sanity_passed:
    raise AssertionError(
        "\nUntrained perplexity is not near vocabulary size.\n"
        "Stop and inspect:\n"
        "  1. logits[:, :-1] must align with tokens[:, 1:]\n"
        "  2. the loss denominator must count only active targets\n"
        "  3. F.cross_entropy must receive raw logits, not softmax\n"
        "  4. the head dimension must equal the tokenizer vocabulary\n"
        "  5. the model must still be randomly initialized\n"
        "  6. initialization must not produce very large logits\n"
        f"Observed PPL={untrained_perplexity.item():.2f}, "
        f"expected near V={V}."
    )


# ---------------------------------------------------------------------
# Save metrics for the README
# ---------------------------------------------------------------------

part1_metrics["perplexity"] = {
    "vocab_size": V,
    "uniform_loss": uniform_loss,
    "untrained_loss": untrained_mean_loss.item(),
    "uniform_perplexity": uniform_perplexity,
    "untrained_perplexity": untrained_perplexity.item(),
    "perplexity_to_vocab_ratio": perplexity_ratio_to_vocab,
    "relative_error": perplexity_relative_error,
    "contributing_targets": untrained_contributing_count,
}

print()
print(
    "STEP 5 PASSED — untrained perplexity is near "
    "the vocabulary size."
)

Uniform-reference sanity check
  Vocabulary size V             : 8,192
  Expected uniform loss ln(V)   : 9.010913
  Observed untrained mean loss  : 9.030918
  Loss difference               : +0.020005 nats

  Expected uniform perplexity   : 8,192.00
  Observed untrained perplexity : 8,357.53
  Perplexity difference         : +165.53
  PPL / V ratio                 : 1.020206
  Relative distance from V      : 2.02%

  Uniform target probability    : 0.00012207
  Observed geometric-mean target probability: 0.00011965
  Contributing targets          : 20

STEP 5 PASSED — untrained perplexity is near the vocabulary size.


In [8]:
# ---------------------------------------------------------------------
# Parameter-count helpers
# ---------------------------------------------------------------------

def count_unique_parameters(*modules: nn.Module) -> int:
    """
    Count Parameter objects once, even when multiple modules reference
    the same tied Parameter.
    """
    seen = set()
    total = 0

    for module in modules:
        for parameter in module.parameters():
            identity = id(parameter)

            if identity not in seen:
                seen.add(identity)
                total += parameter.numel()

    return total


def count_unique_parameter_bytes(*modules: nn.Module) -> int:
    """Count actual parameter-storage bytes without double counting."""
    seen = set()
    total_bytes = 0

    for module in modules:
        for parameter in module.parameters():
            identity = id(parameter)

            if identity not in seen:
                seen.add(identity)
                total_bytes += (
                    parameter.numel()
                    * parameter.element_size()
                )

    return total_bytes


def to_mib(num_bytes: int | float) -> float:
    return num_bytes / 2**20


# ---------------------------------------------------------------------
# Exact head mathematics
# ---------------------------------------------------------------------

head_matrix_parameters = V * D

fp32_bytes_per_parameter = torch.tensor(
    [],
    dtype=torch.float32,
).element_size()

bf16_bytes_per_parameter = torch.tensor(
    [],
    dtype=torch.bfloat16,
).element_size()

fp16_bytes_per_parameter = torch.tensor(
    [],
    dtype=torch.float16,
).element_size()

head_fp32_bytes = (
    head_matrix_parameters
    * fp32_bytes_per_parameter
)

head_bf16_bytes = (
    head_matrix_parameters
    * bf16_bytes_per_parameter
)

head_fp16_bytes = (
    head_matrix_parameters
    * fp16_bytes_per_parameter
)


# ---------------------------------------------------------------------
# Verify that the current model is genuinely tied
# ---------------------------------------------------------------------

assert output_head.weight is model.token_embedding.weight

embedding_pointer = (
    model.token_embedding.weight.data_ptr()
)

tied_head_pointer = (
    output_head.weight.data_ptr()
)

assert embedding_pointer == tied_head_pointer


# ---------------------------------------------------------------------
# Create an untied comparison head on CPU
#
# It has the same shape but owns a separate Parameter object.
# No GPU allocation is needed for this accounting experiment.
# ---------------------------------------------------------------------

untied_output_head = nn.Linear(
    in_features=D,
    out_features=V,
    bias=False,
    device="cpu",
    dtype=torch.float32,
)

assert (
    untied_output_head.weight.shape
    == output_head.weight.shape
)

assert (
    untied_output_head.weight
    is not model.token_embedding.weight
)


# ---------------------------------------------------------------------
# Count unique parameters
# ---------------------------------------------------------------------

trunk_including_embedding_parameters = (
    count_unique_parameters(model)
)

tied_total_parameters = count_unique_parameters(
    model,
    output_head,
)

untied_total_parameters = count_unique_parameters(
    model,
    untied_output_head,
)

tied_embedding_plus_head_unique = (
    head_matrix_parameters
)

untied_embedding_plus_head_unique = (
    2 * head_matrix_parameters
)

tied_additional_head_parameters = 0
untied_additional_head_parameters = (
    head_matrix_parameters
)

parameters_saved_by_tying = (
    untied_total_parameters
    - tied_total_parameters
)


# ---------------------------------------------------------------------
# Exact FP32 parameter-storage bytes
# ---------------------------------------------------------------------

tied_total_parameter_bytes = (
    count_unique_parameter_bytes(
        model,
        output_head,
    )
)

# The CPU comparison head is also FP32, so its actual bytes can be
# added directly to the model's unique parameter bytes.
untied_total_parameter_bytes = (
    count_unique_parameter_bytes(model)
    + count_unique_parameter_bytes(
        untied_output_head
    )
)

bytes_saved_by_tying = (
    untied_total_parameter_bytes
    - tied_total_parameter_bytes
)


# ---------------------------------------------------------------------
# Approximate AdamW training-state cost
#
# For each FP32 parameter:
#   parameter value : 4 bytes
#   gradient        : 4 bytes
#   first moment m  : 4 bytes
#   second moment v : 4 bytes
#
# Total: approximately 16 bytes per parameter.
# This excludes activations, temporary kernels, and allocator overhead.
# ---------------------------------------------------------------------

adamw_fp32_bytes_per_parameter = 16

head_adamw_training_bytes = (
    head_matrix_parameters
    * adamw_fp32_bytes_per_parameter
)


# ---------------------------------------------------------------------
# Correctness assertions
# ---------------------------------------------------------------------

assert head_matrix_parameters == 1_048_576

assert (
    tied_embedding_plus_head_unique
    == 1_048_576
)

assert (
    untied_embedding_plus_head_unique
    == 2_097_152
)

assert (
    parameters_saved_by_tying
    == head_matrix_parameters
)

assert bytes_saved_by_tying == head_fp32_bytes


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Model configuration")
print(f"  Vocabulary size V : {V:,}")
print(f"  Hidden width D    : {D:,}")
print(
    f"  Head shape       : "
    f"[V, D] = [{V:,}, {D:,}]"
)

print()
print("One V x D matrix")
print(
    f"  Parameters       : "
    f"{head_matrix_parameters:,}"
)
print(
    f"  FP32 storage     : "
    f"{to_mib(head_fp32_bytes):.2f} MiB"
)
print(
    f"  BF16 storage     : "
    f"{to_mib(head_bf16_bytes):.2f} MiB"
)
print(
    f"  FP16 storage     : "
    f"{to_mib(head_fp16_bytes):.2f} MiB"
)

print()
print("Embedding + output-head subsystem")
print(
    f"  Tied unique parameters   : "
    f"{tied_embedding_plus_head_unique:,}"
)
print(
    f"  Untied unique parameters : "
    f"{untied_embedding_plus_head_unique:,}"
)
print(
    f"  Tied FP32 storage        : "
    f"{to_mib(head_fp32_bytes):.2f} MiB"
)
print(
    f"  Untied FP32 storage      : "
    f"{to_mib(2 * head_fp32_bytes):.2f} MiB"
)

print()
print("Entire observable model")
print(
    f"  Trunk including embedding: "
    f"{trunk_including_embedding_parameters:,}"
)
print(
    f"  Tied total parameters    : "
    f"{tied_total_parameters:,}"
)
print(
    f"  Untied total parameters  : "
    f"{untied_total_parameters:,}"
)
print(
    f"  Parameters saved by tying: "
    f"{parameters_saved_by_tying:,}"
)

print()
print("Entire-model FP32 parameter storage")
print(
    f"  Tied model   : "
    f"{to_mib(tied_total_parameter_bytes):.2f} MiB"
)
print(
    f"  Untied model : "
    f"{to_mib(untied_total_parameter_bytes):.2f} MiB"
)
print(
    f"  Saved        : "
    f"{to_mib(bytes_saved_by_tying):.2f} MiB"
)

print()
print("Approximate AdamW training-state effect")
print(
    f"  One additional untied V x D matrix costs "
    f"approximately "
    f"{to_mib(head_adamw_training_bytes):.2f} MiB"
)
print(
    "  This includes FP32 parameter, gradient, "
    "first moment, and second moment."
)


# ---------------------------------------------------------------------
# Save metrics for the final README
# ---------------------------------------------------------------------

part1_metrics["weight_tying"] = {
    "vocab_size": V,
    "hidden_size": D,
    "head_matrix_parameters": head_matrix_parameters,
    "tied_embedding_head_unique_parameters": (
        tied_embedding_plus_head_unique
    ),
    "untied_embedding_head_unique_parameters": (
        untied_embedding_plus_head_unique
    ),
    "tied_total_parameters": tied_total_parameters,
    "untied_total_parameters": untied_total_parameters,
    "parameters_saved": parameters_saved_by_tying,
    "fp32_mib_saved": to_mib(bytes_saved_by_tying),
    "estimated_adamw_training_mib_saved": to_mib(
        head_adamw_training_bytes
    ),
}

# The comparison head is no longer needed.
del untied_output_head

print()
print("STEP 6 PASSED")

Model configuration
  Vocabulary size V : 8,192
  Hidden width D    : 128
  Head shape       : [V, D] = [8,192, 128]

One V x D matrix
  Parameters       : 1,048,576
  FP32 storage     : 4.00 MiB
  BF16 storage     : 2.00 MiB
  FP16 storage     : 2.00 MiB

Embedding + output-head subsystem
  Tied unique parameters   : 1,048,576
  Untied unique parameters : 2,097,152
  Tied FP32 storage        : 4.00 MiB
  Untied FP32 storage      : 8.00 MiB

Entire observable model
  Trunk including embedding: 1,509,888
  Tied total parameters    : 1,509,888
  Untied total parameters  : 2,558,464
  Parameters saved by tying: 1,048,576

Entire-model FP32 parameter storage
  Tied model   : 5.76 MiB
  Untied model : 9.76 MiB
  Saved        : 4.00 MiB

Approximate AdamW training-state effect
  One additional untied V x D matrix costs approximately 16.00 MiB
  This includes FP32 parameter, gradient, first moment, and second moment.

STEP 6 PASSED


In [9]:
import gc
import time


# ---------------------------------------------------------------------
# Benchmark configuration
# ---------------------------------------------------------------------

BENCHMARK_TOKENS = 16_384
CHUNK_SIZE = 512
BENCHMARK_DTYPE = torch.float32

assert BENCHMARK_TOKENS % CHUNK_SIZE == 0
assert DEVICE.type == "cuda"

bytes_per_value = torch.tensor(
    [],
    dtype=BENCHMARK_DTYPE,
).element_size()

materialized_logit_bytes = (
    BENCHMARK_TOKENS
    * V
    * bytes_per_value
)

chunk_logit_bytes = (
    CHUNK_SIZE
    * V
    * bytes_per_value
)

theoretical_logit_ratio = (
    materialized_logit_bytes
    / chunk_logit_bytes
)


print("Benchmark configuration")
print(
    f"  GPU                   : "
    f"{torch.cuda.get_device_name(DEVICE)}"
)
print(
    f"  Hidden shape          : "
    f"[N, D] = "
    f"[{BENCHMARK_TOKENS:,}, {D:,}]"
)
print(
    f"  Weight shape          : "
    f"[V, D] = [{V:,}, {D:,}]"
)
print(
    f"  Full logits shape     : "
    f"[N, V] = "
    f"[{BENCHMARK_TOKENS:,}, {V:,}]"
)
print(
    f"  Chunk logits shape    : "
    f"[C, V] = [{CHUNK_SIZE:,}, {V:,}]"
)
print(
    f"  Raw full logits       : "
    f"{to_mib(materialized_logit_bytes):.2f} MiB"
)
print(
    f"  Raw chunk logits      : "
    f"{to_mib(chunk_logit_bytes):.2f} MiB"
)
print(
    f"  Theoretical logit ratio: "
    f"{theoretical_logit_ratio:.2f}x"
)


# ---------------------------------------------------------------------
# Fixed benchmark inputs
#
# These templates remain identical across both measurements. Each
# method receives a fresh leaf clone so gradients cannot leak between
# experiments.
# ---------------------------------------------------------------------

torch.manual_seed(6202026)
torch.cuda.manual_seed_all(6202026)

hidden_template = torch.randn(
    BENCHMARK_TOKENS,
    D,
    device=DEVICE,
    dtype=BENCHMARK_DTYPE,
)

weight_template = torch.randn(
    V,
    D,
    device=DEVICE,
    dtype=BENCHMARK_DTYPE,
) * 0.02

target_template = torch.randint(
    low=0,
    high=V,
    size=(BENCHMARK_TOKENS,),
    device=DEVICE,
    dtype=torch.long,
)


# ---------------------------------------------------------------------
# Method A: materialized cross-entropy
# ---------------------------------------------------------------------

def materialized_cross_entropy_backward(
    hidden: torch.Tensor,
    weight: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    """
    Materialize all [N, V] logits, compute mean CE, and backpropagate.
    """
    logits = F.linear(hidden, weight)

    loss = F.cross_entropy(
        logits,
        targets,
        reduction="mean",
    )

    loss.backward()

    return loss.detach()


# ---------------------------------------------------------------------
# Method B: custom chunked cross-entropy
# ---------------------------------------------------------------------

def chunked_cross_entropy_backward(
    hidden: torch.Tensor,
    weight: torch.Tensor,
    targets: torch.Tensor,
    chunk_size: int,
) -> torch.Tensor:
    """
    Compute exactly the same mean cross-entropy in token chunks.

    Each chunk performs backward immediately. Gradients accumulate in
    hidden.grad and weight.grad, while the chunk's [C, V] logits can be
    freed before the next iteration.
    """
    token_count = targets.numel()

    if hidden.ndim != 2:
        raise ValueError(
            f"hidden must be [N, D], got {tuple(hidden.shape)}"
        )

    if weight.ndim != 2:
        raise ValueError(
            f"weight must be [V, D], got {tuple(weight.shape)}"
        )

    if targets.ndim != 1:
        raise ValueError(
            f"targets must be [N], got {tuple(targets.shape)}"
        )

    if hidden.shape[0] != token_count:
        raise ValueError(
            "hidden and targets must contain the same token count"
        )

    if hidden.shape[1] != weight.shape[1]:
        raise ValueError(
            "hidden width must equal output-head width"
        )

    # FP64 scalar accumulation makes the reported loss insensitive to
    # the order in which chunk sums are combined. The expensive logits
    # remain in the configured benchmark dtype.
    detached_loss_sum = torch.zeros(
        (),
        dtype=torch.float64,
        device=hidden.device,
    )

    for start in range(0, token_count, chunk_size):
        end = min(start + chunk_size, token_count)

        chunk_hidden = hidden[start:end]
        chunk_targets = targets[start:end]

        # Only [chunk_size, V] logits exist in this iteration.
        chunk_logits = F.linear(
            chunk_hidden,
            weight,
        )

        chunk_loss_sum = F.cross_entropy(
            chunk_logits,
            chunk_targets,
            reduction="sum",
        )

        # Scaling every chunk by the global token count makes the
        # accumulated gradient equal to the gradient of one global mean:
        #
        # ∇[(Σ chunk losses) / N]
        (chunk_loss_sum / token_count).backward()

        detached_loss_sum += (
            chunk_loss_sum.detach().double()
        )

    return (
        detached_loss_sum / token_count
    ).to(dtype=hidden.dtype)


# ---------------------------------------------------------------------
# Warm up CUDA kernels outside the measurements
# ---------------------------------------------------------------------

warm_hidden = torch.randn(
    32,
    D,
    device=DEVICE,
    requires_grad=True,
)

warm_weight = torch.randn(
    V,
    D,
    device=DEVICE,
    requires_grad=True,
)

warm_targets = torch.randint(
    0,
    V,
    (32,),
    device=DEVICE,
)

warm_loss = F.cross_entropy(
    F.linear(warm_hidden, warm_weight),
    warm_targets,
)

warm_loss.backward()
torch.cuda.synchronize()

del (
    warm_hidden,
    warm_weight,
    warm_targets,
    warm_loss,
)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()


# ---------------------------------------------------------------------
# Measurement A: materialized forward + backward
# ---------------------------------------------------------------------

materialized_hidden = (
    hidden_template.detach().clone().requires_grad_(True)
)

materialized_weight = (
    weight_template.detach().clone().requires_grad_(True)
)

torch.cuda.synchronize()
gc.collect()
torch.cuda.empty_cache()

materialized_baseline_bytes = (
    torch.cuda.memory_allocated(DEVICE)
)

torch.cuda.reset_peak_memory_stats(DEVICE)

materialized_start = time.perf_counter()

materialized_loss = materialized_cross_entropy_backward(
    materialized_hidden,
    materialized_weight,
    target_template,
)

torch.cuda.synchronize()

materialized_seconds = (
    time.perf_counter() - materialized_start
)

materialized_absolute_peak_bytes = (
    torch.cuda.max_memory_allocated(DEVICE)
)

materialized_incremental_peak_bytes = (
    materialized_absolute_peak_bytes
    - materialized_baseline_bytes
)

# Keep CPU copies for exact gradient comparison.
materialized_hidden_gradient = (
    materialized_hidden.grad.detach().cpu()
)

materialized_weight_gradient = (
    materialized_weight.grad.detach().cpu()
)

materialized_loss_value = (
    materialized_loss.item()
)

del (
    materialized_hidden,
    materialized_weight,
    materialized_loss,
)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()


# ---------------------------------------------------------------------
# Measurement B: chunked forward + backward
# ---------------------------------------------------------------------

chunked_hidden = (
    hidden_template.detach().clone().requires_grad_(True)
)

chunked_weight = (
    weight_template.detach().clone().requires_grad_(True)
)

torch.cuda.synchronize()
gc.collect()
torch.cuda.empty_cache()

chunked_baseline_bytes = (
    torch.cuda.memory_allocated(DEVICE)
)

torch.cuda.reset_peak_memory_stats(DEVICE)

chunked_start = time.perf_counter()

chunked_loss = chunked_cross_entropy_backward(
    chunked_hidden,
    chunked_weight,
    target_template,
    chunk_size=CHUNK_SIZE,
)

torch.cuda.synchronize()

chunked_seconds = (
    time.perf_counter() - chunked_start
)

chunked_absolute_peak_bytes = (
    torch.cuda.max_memory_allocated(DEVICE)
)

chunked_incremental_peak_bytes = (
    chunked_absolute_peak_bytes
    - chunked_baseline_bytes
)

chunked_hidden_gradient = (
    chunked_hidden.grad.detach().cpu()
)

chunked_weight_gradient = (
    chunked_weight.grad.detach().cpu()
)

chunked_loss_value = chunked_loss.item()


# ---------------------------------------------------------------------
# Numerical-equivalence checks
# ---------------------------------------------------------------------

loss_absolute_difference = abs(
    materialized_loss_value
    - chunked_loss_value
)

hidden_gradient_max_difference = (
    materialized_hidden_gradient
    - chunked_hidden_gradient
).abs().max().item()

weight_gradient_max_difference = (
    materialized_weight_gradient
    - chunked_weight_gradient
).abs().max().item()

assert math.isclose(
    materialized_loss_value,
    chunked_loss_value,
    rel_tol=1e-6,
    abs_tol=1e-5,
)

torch.testing.assert_close(
    materialized_hidden_gradient,
    chunked_hidden_gradient,
    rtol=2e-4,
    atol=2e-6,
)

torch.testing.assert_close(
    materialized_weight_gradient,
    chunked_weight_gradient,
    rtol=2e-4,
    atol=2e-6,
)


# ---------------------------------------------------------------------
# Memory comparison
# ---------------------------------------------------------------------

memory_reduction_ratio = (
    materialized_incremental_peak_bytes
    / chunked_incremental_peak_bytes
)

memory_saved_bytes = (
    materialized_incremental_peak_bytes
    - chunked_incremental_peak_bytes
)


print()
print("Numerical equivalence")
print(
    f"  Materialized loss       : "
    f"{materialized_loss_value:.8f}"
)
print(
    f"  Chunked loss            : "
    f"{chunked_loss_value:.8f}"
)
print(
    f"  Absolute loss difference: "
    f"{loss_absolute_difference:.10f}"
)
print(
    f"  Max hidden-grad diff    : "
    f"{hidden_gradient_max_difference:.3e}"
)
print(
    f"  Max weight-grad diff    : "
    f"{weight_gradient_max_difference:.3e}"
)

print()
print("Measured CUDA peak allocation above baseline")
print(
    f"  Materialized CE peak : "
    f"{to_mib(materialized_incremental_peak_bytes):,.2f} MiB "
    f"({materialized_incremental_peak_bytes / 2**30:.3f} GiB)"
)
print(
    f"  Chunked CE peak      : "
    f"{to_mib(chunked_incremental_peak_bytes):,.2f} MiB "
    f"({chunked_incremental_peak_bytes / 2**30:.3f} GiB)"
)
print(
    f"  Memory saved         : "
    f"{to_mib(memory_saved_bytes):,.2f} MiB"
)
print(
    f"  Measured reduction   : "
    f"{memory_reduction_ratio:.2f}x"
)

print()
print("Runtime diagnostic")
print(
    f"  Materialized time : "
    f"{materialized_seconds:.3f} s"
)
print(
    f"  Chunked time      : "
    f"{chunked_seconds:.3f} s"
)


# ---------------------------------------------------------------------
# Save metrics for the README
# ---------------------------------------------------------------------

part1_metrics["memory"] = {
    "benchmark_tokens": BENCHMARK_TOKENS,
    "chunk_size": CHUNK_SIZE,
    "dtype": str(BENCHMARK_DTYPE),
    "raw_materialized_logits_mib": to_mib(
        materialized_logit_bytes
    ),
    "raw_chunk_logits_mib": to_mib(
        chunk_logit_bytes
    ),
    "materialized_peak_mib": to_mib(
        materialized_incremental_peak_bytes
    ),
    "chunked_peak_mib": to_mib(
        chunked_incremental_peak_bytes
    ),
    "measured_reduction_ratio": memory_reduction_ratio,
    "materialized_loss": materialized_loss_value,
    "chunked_loss": chunked_loss_value,
    "absolute_loss_difference": loss_absolute_difference,
    "materialized_seconds": materialized_seconds,
    "chunked_seconds": chunked_seconds,
}

assert (
    chunked_incremental_peak_bytes
    < materialized_incremental_peak_bytes
)

print()
print("STEP 7 PASSED")


# ---------------------------------------------------------------------
# Release benchmark-only allocations
# ---------------------------------------------------------------------

del (
    chunked_hidden,
    chunked_weight,
    chunked_loss,
    hidden_template,
    weight_template,
    target_template,
    materialized_hidden_gradient,
    materialized_weight_gradient,
    chunked_hidden_gradient,
    chunked_weight_gradient,
)

gc.collect()
torch.cuda.empty_cache()

Benchmark configuration
  GPU                   : NVIDIA GeForce RTX 3070 Laptop GPU
  Hidden shape          : [N, D] = [16,384, 128]
  Weight shape          : [V, D] = [8,192, 128]
  Full logits shape     : [N, V] = [16,384, 8,192]
  Chunk logits shape    : [C, V] = [512, 8,192]
  Raw full logits       : 512.00 MiB
  Raw chunk logits      : 16.00 MiB
  Theoretical logit ratio: 32.00x



Numerical equivalence
  Materialized loss       : 9.03569412
  Chunked loss            : 9.03569508
  Absolute loss difference: 0.0000009537
  Max hidden-grad diff    : 2.638e-11
  Max weight-grad diff    : 1.659e-09

Measured CUDA peak allocation above baseline
  Materialized CE peak : 2,048.00 MiB (2.000 GiB)
  Chunked CE peak      : 76.00 MiB (0.074 GiB)
  Memory saved         : 1,972.00 MiB
  Measured reduction   : 26.95x

Runtime diagnostic
  Materialized time : 0.075 s
  Chunked time      : 0.031 s

STEP 7 PASSED


In [10]:
# ---------------------------------------------------------------------
# MTP configuration
# ---------------------------------------------------------------------

MTP_SEQUENCE_LENGTH = 256
MTP_SEQUENCE_COUNT = 24
MTP_BATCH_SIZE = 4
MTP_TRAIN_STEPS = 80
MTP_LOG_INTERVAL = 10
MTP_LEARNING_RATE = 1e-3

torch.manual_seed(9202026)
torch.cuda.manual_seed_all(9202026)


# ---------------------------------------------------------------------
# Select dense, real Session 6 training sequences
# ---------------------------------------------------------------------

mtp_rows = []

with gzip.open(SEQUENCES_PATH, "rt", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)

        if (
            row["lane"] == "general"
            and int(row["sequence_length"]) == MTP_SEQUENCE_LENGTH
            and int(row["loss_bearing_tokens"]) >= 200
        ):
            mtp_rows.append(row)

        if len(mtp_rows) == MTP_SEQUENCE_COUNT:
            break

assert len(mtp_rows) == MTP_SEQUENCE_COUNT, (
    f"Found only {len(mtp_rows)} suitable MTP sequences"
)

mtp_arrays = [
    load_sequence(row)
    for row in mtp_rows
]

mtp_tokens = torch.as_tensor(
    np.stack([row["input_ids"] for row in mtp_arrays]),
    dtype=torch.long,
    device=DEVICE,
)

mtp_loss_mask = torch.as_tensor(
    np.stack([row["loss_mask"] for row in mtp_arrays]),
    dtype=torch.bool,
    device=DEVICE,
)

mtp_segment_ids = torch.as_tensor(
    np.stack([row["segment_ids"] for row in mtp_arrays]),
    dtype=torch.long,
    device=DEVICE,
)

mtp_position_ids = torch.as_tensor(
    np.stack([row["position_ids"] for row in mtp_arrays]),
    dtype=torch.long,
    device=DEVICE,
)

assert mtp_tokens.shape == (
    MTP_SEQUENCE_COUNT,
    MTP_SEQUENCE_LENGTH,
)


# ---------------------------------------------------------------------
# Second output head: h[t] -> token[t+2]
#
# It must be independent of Head 1. If it shared the exact same weight
# matrix, both heads would always produce the same logits from h[t].
# ---------------------------------------------------------------------

mtp_head2 = nn.Linear(
    in_features=D,
    out_features=V,
    bias=False,
).to(DEVICE)

nn.init.normal_(
    mtp_head2.weight,
    mean=0.0,
    std=0.02,
)

assert mtp_head2.weight is not output_head.weight
assert mtp_head2.weight.shape == (V, D)

mtp_head2_parameter_count = (
    mtp_head2.weight.numel()
)

assert (
    mtp_head2_parameter_count
    == V * D
)


# ---------------------------------------------------------------------
# Construct target-aligned Head 1 and Head 2 masks
# ---------------------------------------------------------------------

def build_mtp_masks(
    loss_mask: torch.Tensor,
    segment_ids: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Returns:
        head1_mask [B, T-1]
        head2_mask [B, T-2]
    """

    # Head 1: h[t] predicts x[t+1].
    head1_mask = (
        loss_mask[:, 1:]
        & (segment_ids[:, :-1] >= 0)
        & (segment_ids[:, 1:] >= 0)
        & (
            segment_ids[:, :-1]
            == segment_ids[:, 1:]
        )
    )

    # Head 2: h[t] predicts x[t+2].
    #
    # Require the complete t -> t+1 -> t+2 span to remain inside
    # one document. This rejects padding and either possible boundary.
    segment_t = segment_ids[:, :-2]
    segment_t_plus_1 = segment_ids[:, 1:-1]
    segment_t_plus_2 = segment_ids[:, 2:]

    head2_mask = (
        loss_mask[:, 2:]
        & (segment_t >= 0)
        & (segment_t_plus_1 >= 0)
        & (segment_t_plus_2 >= 0)
        & (segment_t == segment_t_plus_1)
        & (segment_t_plus_1 == segment_t_plus_2)
    )

    return head1_mask, head2_mask


# ---------------------------------------------------------------------
# Compute both losses from one shared decoder forward pass
# ---------------------------------------------------------------------

def compute_mtp_losses(
    batch_tokens: torch.Tensor,
    batch_loss_mask: torch.Tensor,
    batch_segment_ids: torch.Tensor,
    batch_position_ids: torch.Tensor,
) -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    int,
    int,
]:
    embeddings = (
        model.token_embedding(batch_tokens)
        + model.position_embedding(batch_position_ids)
    )

    hidden = model.forward_from_embeddings(
        embeddings,
        batch_segment_ids,
    )

    head1_mask, head2_mask = build_mtp_masks(
        batch_loss_mask,
        batch_segment_ids,
    )

    # Head 1 alignment:
    # hidden[:, t] -> tokens[:, t+1]
    head1_hidden = hidden[:, :-1, :][head1_mask]
    head1_targets = batch_tokens[:, 1:][head1_mask]

    # Head 2 alignment:
    # hidden[:, t] -> tokens[:, t+2]
    head2_hidden = hidden[:, :-2, :][head2_mask]
    head2_targets = batch_tokens[:, 2:][head2_mask]

    if head1_targets.numel() == 0:
        raise ValueError("Head 1 has no active targets")

    if head2_targets.numel() == 0:
        raise ValueError("Head 2 has no active targets")

    # Project only active hidden states. This avoids producing logits
    # for padding, boundaries, and origin-masked targets.
    head1_logits = output_head(head1_hidden)
    head2_logits = mtp_head2(head2_hidden)

    head1_loss = F.cross_entropy(
        head1_logits,
        head1_targets,
        reduction="mean",
    )

    head2_loss = F.cross_entropy(
        head2_logits,
        head2_targets,
        reduction="mean",
    )

    total_loss = head1_loss + head2_loss

    return (
        head1_loss,
        head2_loss,
        total_loss,
        int(head1_targets.numel()),
        int(head2_targets.numel()),
    )


# ---------------------------------------------------------------------
# Print real string-level MTP alignment before training
# ---------------------------------------------------------------------

all_head1_mask, all_head2_mask = build_mtp_masks(
    mtp_loss_mask,
    mtp_segment_ids,
)

# Head 1's first T-2 mask entries correspond to the same h[t]
# positions as Head 2.
both_heads_active = (
    all_head1_mask[:, :-1]
    & all_head2_mask
)

active_locations = torch.nonzero(
    both_heads_active,
    as_tuple=False,
)

assert len(active_locations) >= 8

print("String-level MTP alignment")
print(
    "The same hidden state h[t] feeds both prediction heads."
)
print()

header = (
    f"{'t':>4} | "
    f"{'INPUT x[t]':<22} | "
    f"{'HEAD 1 x[t+1]':<22} | "
    f"{'HEAD 2 x[t+2]':<22}"
)

print(header)
print("-" * len(header))

for batch_index, t in active_locations[:8].tolist():
    input_text = repr(
        token_string(
            mtp_tokens[batch_index, t].item()
        )
    )

    head1_text = repr(
        token_string(
            mtp_tokens[batch_index, t + 1].item()
        )
    )

    head2_text = repr(
        token_string(
            mtp_tokens[batch_index, t + 2].item()
        )
    )

    print(
        f"{t:>4} | "
        f"{input_text:<22} | "
        f"{head1_text:<22} | "
        f"{head2_text:<22}"
    )


# ---------------------------------------------------------------------
# Unique optimizer parameters
#
# output_head.weight is tied to model.token_embedding.weight. Passing
# both without deduplication would give the optimizer the same Parameter
# twice.
# ---------------------------------------------------------------------

def unique_trainable_parameters(
    *modules: nn.Module,
) -> list[nn.Parameter]:
    parameters = []
    seen = set()

    for module in modules:
        for parameter in module.parameters():
            if not parameter.requires_grad:
                continue

            identity = id(parameter)

            if identity not in seen:
                seen.add(identity)
                parameters.append(parameter)

    return parameters


training_parameters = unique_trainable_parameters(
    model,
    output_head,
    mtp_head2,
)

assert len({
    id(parameter)
    for parameter in training_parameters
}) == len(training_parameters)

optimizer = torch.optim.AdamW(
    training_parameters,
    lr=MTP_LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.01,
)


# ---------------------------------------------------------------------
# Evaluation over all 24 selected sequences
# ---------------------------------------------------------------------

@torch.no_grad()
def evaluate_mtp_dataset() -> tuple[float, float, float, int, int]:
    model.eval()
    output_head.eval()
    mtp_head2.eval()

    head1_loss_sum = 0.0
    head2_loss_sum = 0.0
    head1_count_sum = 0
    head2_count_sum = 0

    for start in range(
        0,
        MTP_SEQUENCE_COUNT,
        MTP_BATCH_SIZE,
    ):
        end = start + MTP_BATCH_SIZE

        loss1, loss2, _, count1, count2 = (
            compute_mtp_losses(
                mtp_tokens[start:end],
                mtp_loss_mask[start:end],
                mtp_segment_ids[start:end],
                mtp_position_ids[start:end],
            )
        )

        head1_loss_sum += loss1.item() * count1
        head2_loss_sum += loss2.item() * count2
        head1_count_sum += count1
        head2_count_sum += count2

    mean_head1 = (
        head1_loss_sum / head1_count_sum
    )

    mean_head2 = (
        head2_loss_sum / head2_count_sum
    )

    return (
        mean_head1,
        mean_head2,
        mean_head1 + mean_head2,
        head1_count_sum,
        head2_count_sum,
    )


# ---------------------------------------------------------------------
# Initial evaluation
# ---------------------------------------------------------------------

training_history = []

initial_metrics = evaluate_mtp_dataset()

training_history.append({
    "step": 0,
    "head1_loss": initial_metrics[0],
    "head2_loss": initial_metrics[1],
    "total_loss": initial_metrics[2],
})


# ---------------------------------------------------------------------
# Short joint-training loop
# ---------------------------------------------------------------------

for step in range(1, MTP_TRAIN_STEPS + 1):
    model.train()
    output_head.train()
    mtp_head2.train()

    # Deterministic cyclic mini-batches.
    batch_start = (
        (step - 1) * MTP_BATCH_SIZE
    ) % MTP_SEQUENCE_COUNT

    batch_indices = torch.arange(
        batch_start,
        batch_start + MTP_BATCH_SIZE,
        device=DEVICE,
    ) % MTP_SEQUENCE_COUNT

    optimizer.zero_grad(set_to_none=True)

    loss1, loss2, total_loss, _, _ = (
        compute_mtp_losses(
            mtp_tokens[batch_indices],
            mtp_loss_mask[batch_indices],
            mtp_segment_ids[batch_indices],
            mtp_position_ids[batch_indices],
        )
    )

    # ∇L_total = ∇L_1 + ∇L_2
    total_loss.backward()

    gradient_norm = torch.nn.utils.clip_grad_norm_(
        training_parameters,
        max_norm=1.0,
    )

    optimizer.step()

    if (
        step == 1
        or step % MTP_LOG_INTERVAL == 0
        or step == MTP_TRAIN_STEPS
    ):
        evaluation = evaluate_mtp_dataset()

        training_history.append({
            "step": step,
            "head1_loss": evaluation[0],
            "head2_loss": evaluation[1],
            "total_loss": evaluation[2],
            "gradient_norm": float(gradient_norm),
        })


# ---------------------------------------------------------------------
# Report loss trajectories
# ---------------------------------------------------------------------

print()
print("MTP training history")
print(
    f"{'step':>5} | "
    f"{'Head 1: t+1':>14} | "
    f"{'Head 2: t+2':>14} | "
    f"{'sum':>14} | "
    f"{'Head2-Head1':>14}"
)
print("-" * 73)

for row in training_history:
    gap = (
        row["head2_loss"]
        - row["head1_loss"]
    )

    print(
        f"{row['step']:>5} | "
        f"{row['head1_loss']:>14.6f} | "
        f"{row['head2_loss']:>14.6f} | "
        f"{row['total_loss']:>14.6f} | "
        f"{gap:>+14.6f}"
    )


final_metrics = evaluate_mtp_dataset()

initial_head1 = initial_metrics[0]
initial_head2 = initial_metrics[1]

final_head1 = final_metrics[0]
final_head2 = final_metrics[1]
final_total = final_metrics[2]

head1_reduction = initial_head1 - final_head1
head2_reduction = initial_head2 - final_head2
final_gap = final_head2 - final_head1


print()
print("PART 2 SMOKE-TEST RESULTS (80-update demonstration only)")
print(
    f"  Head 1 loss, t+1 : "
    f"{final_head1:.6f}"
)
print(
    f"  Head 2 loss, t+2 : "
    f"{final_head2:.6f}"
)
print(
    f"  Combined loss    : "
    f"{final_total:.6f}"
)

print()
print("Change over training")
print(
    f"  Head 1 reduction : "
    f"{head1_reduction:.6f}"
)
print(
    f"  Head 2 reduction : "
    f"{head2_reduction:.6f}"
)
print(
    f"  Final H2-H1 gap  : "
    f"{final_gap:+.6f}"
)

print()
print("Target accounting")
print(
    f"  Head 1 targets   : "
    f"{final_metrics[3]:,}"
)
print(
    f"  Head 2 targets   : "
    f"{final_metrics[4]:,}"
)
print(
    f"  Head 2 parameters: "
    f"{mtp_head2_parameter_count:,} "
    f"({to_mib(mtp_head2_parameter_count * 4):.2f} MiB FP32)"
)


# ---------------------------------------------------------------------
# Save Part 2 smoke-test metrics
# ---------------------------------------------------------------------

part2_metrics = {
    "result_scope": "in_notebook_smoke_test",
    "head1_initial_loss": initial_head1,
    "head2_initial_loss": initial_head2,
    "head1_final_loss": final_head1,
    "head2_final_loss": final_head2,
    "combined_final_loss": final_total,
    "head1_reduction": head1_reduction,
    "head2_reduction": head2_reduction,
    "final_head2_minus_head1": final_gap,
    "head1_target_count": final_metrics[3],
    "head2_target_count": final_metrics[4],
    "head2_parameters": mtp_head2_parameter_count,
    "history": training_history,
}

assert math.isclose(
    final_total,
    final_head1 + final_head2,
    rel_tol=0.0,
    abs_tol=1e-10,
)

print()
print("STEP 8 PASSED")

String-level MTP alignment
The same hidden state h[t] feeds both prediction heads.

   t | INPUT x[t]             | HEAD 1 x[t+1]          | HEAD 2 x[t+2]         
-------------------------------------------------------------------------------
   0 | '<bos>'                | 'Er'                   | 'win'                 
   1 | 'Er'                   | 'win'                  | '<0x20>'              
   2 | 'win'                  | '<0x20>'               | 'St'                  
   3 | '<0x20>'               | 'St'                   | 'off'                 
   4 | 'St'                   | 'off'                  | '\\n\\n'              
   5 | 'off'                  | '\\n\\n'               | 'Er'                  
   6 | '\\n\\n'               | 'Er'                   | 'win'                 
   7 | 'Er'                   | 'win'                  | '<0x20>'              



MTP training history
 step |    Head 1: t+1 |    Head 2: t+2 |            sum |    Head2-Head1
-------------------------------------------------------------------------
    0 |       9.040891 |       9.086224 |      18.127115 |      +0.045334
    1 |       8.601353 |       8.535919 |      17.137271 |      -0.065434
   10 |       7.192683 |       7.199342 |      14.392025 |      +0.006659
   20 |       5.968294 |       6.006295 |      11.974589 |      +0.038002
   30 |       5.210300 |       5.258846 |      10.469146 |      +0.048546
   40 |       4.779070 |       4.886053 |       9.665123 |      +0.106983
   50 |       4.615221 |       4.770145 |       9.385366 |      +0.154924
   60 |       4.464465 |       4.642294 |       9.106759 |      +0.177829
   70 |       4.354725 |       4.563157 |       8.917882 |      +0.208431
   80 |       4.244803 |       4.448154 |       8.692957 |      +0.203351

PART 2 SMOKE-TEST RESULTS (80-update demonstration only)
  Head 1 loss, t+1 : 4.244803
  

In [11]:
from pathlib import Path
import json
import textwrap


required = {
    "padding",
    "boundary",
    "perplexity",
    "weight_tying",
    "memory",
}

missing = required - set(part1_metrics)
assert not missing, f"Missing Part 1 metrics: {sorted(missing)}"
assert "part2_metrics" in globals()

p = part1_metrics["padding"]
b = part1_metrics["boundary"]
pp = part1_metrics["perplexity"]
wt = part1_metrics["weight_tying"]
mem = part1_metrics["memory"]
mtp = part2_metrics

history = mtp["history"]
later = [row for row in history if row["step"] >= 10]
head2_higher = sum(
    row["head2_loss"] > row["head1_loss"]
    for row in later
)

submission_metrics = {
    "hardware": {
        "gpu": torch.cuda.get_device_name(DEVICE),
        "pytorch": torch.__version__,
    },
    "configuration": {
        "B": int(B),
        "T": int(T),
        "D": D,
        "V": V,
        "layers": NUM_LAYERS,
        "attention_heads": NUM_HEADS,
        "feed_forward_size": FFN_SIZE,
    },
    "part1": {
        "tensor_flow": {
            "tokens": list(tokens.shape),
            "embeddings": list(embeddings.shape),
            "hidden": list(hidden.shape),
            "logits": list(logits.shape),
            "shifted_logits": list(prediction_logits.shape),
            "targets": list(targets.shape),
            "logits_mib": logits_mib,
        },
        "string_shift": {
            "relationship": "h[t] predicts x[t+1]",
            "boundary_pair": (
                f"{boundary_input_string} -> "
                f"{boundary_target_string}"
            ),
            "boundary_position": removed_t,
        },
        "padding": p,
        "boundary": b,
        "perplexity": pp,
        "weight_tying": wt,
        "memory": mem,
    },
    "part2": mtp,
}

mtp_analysis = (
    f"Head 2 started {history[0]['head2_loss'] - history[0]['head1_loss']:+.6f} "
    f"nats relative to Head 1. It briefly crossed below Head 1 after "
    f"the first update, but was higher at {head2_higher} of "
    f"{len(later)} logged checkpoints from step 10 onward. It ended "
    f"{mtp['final_head2_minus_head1']:+.6f} nats above Head 1. "
    f"Head 1 improved by {mtp['head1_reduction']:.6f} nats and "
    f"Head 2 improved by {mtp['head2_reduction']:.6f} nats. This is "
    f"consistent with t+2 having greater conditional uncertainty: "
    f"h[t] must predict x[t+2] without observing x[t+1]. Head 2 also "
    f"had {mtp['head1_target_count'] - mtp['head2_target_count']} fewer "
    f"valid targets because each document loses one additional "
    f"prediction position. These are training-subset measurements, "
    f"not held-out generalization results."
)

readme = f"""
# Assignment 9: Observable Cross-Entropy and Multi-Token Prediction

This notebook implements and verifies a shifted next-token
cross-entropy harness, padding and document-boundary masks, a
perplexity sanity check, tied versus untied output heads, custom
chunked cross-entropy, and a second output head predicting `t+2`.

## Configuration

| Item | Value |
|---|---:|
| GPU | {torch.cuda.get_device_name(DEVICE)} |
| PyTorch | {torch.__version__} |
| Vocabulary size `V` | {V:,} |
| Hidden width `D` | {D:,} |
| Decoder layers | {NUM_LAYERS} |
| Attention heads | {NUM_HEADS} |
| Feed-forward width | {FFN_SIZE:,} |
| Profiling dtype | FP32 |

## Part 1: Seven Harness Checks

### 1. Observable tensor flow

| Tensor | Shape |
|---|---:|
| Tokens | `{tuple(tokens.shape)}` |
| Embeddings | `{tuple(embeddings.shape)}` |
| Hidden states | `{tuple(hidden.shape)}` |
| Logits | `{tuple(logits.shape)}` |
| Shifted logits | `{tuple(prediction_logits.shape)}` |
| Shifted targets | `{tuple(targets.shape)}` |

`B={B}` is batch size, `T={T}` is sequence length, `D={D}` is hidden
width, and `V={V}` is vocabulary size. The observed full logits occupied
**{logits_mib:.2f} MiB** in FP32.

### 2. String-level shift verification

The harness uses `logits[:, :-1]` to predict `tokens[:, 1:]`, establishing
`h[t] -> x[t+1]`.

The actual decoded boundary pair was:

    {boundary_input_string!r} -> {boundary_target_string!r}

It occurred at prediction position `{removed_t}` and was excluded from
the loss.

### 3. Padding mask

| Measurement | Value |
|---|---:|
| Targets before masking | {p['count_before']:,} |
| Padding targets removed | {p['padding_targets_removed']:,} |
| Targets after masking | {p['count_after']:,} |
| Naive loss | {p['naive_loss']:.6f} |
| Masked loss | {p['masked_loss']:.6f} |

The contributing-token count changed from **{p['count_before']} to
{p['count_after']}**. Masking is based on the shifted target, so the
first real-token-to-padding transition is also removed.

### 4. Packed-document boundary

The packed transition was `{b['token_pair']}`.

| Measurement | Before | After |
|---|---:|---:|
| Loss | {b['loss_before']:.6f} | {b['loss_after']:.6f} |
| Contributing targets | {b['count_before']:,} | {b['count_after']:,} |

The boundary token had loss **{b['boundary_token_loss']:.6f}**. Removing
it changed the mean by **{b['loss_change_after_minus_before']:+.6f}**.

It was removed because the two documents are independent, not because
masking is guaranteed to lower the reported loss.

### 5. Perplexity sanity check

Perplexity is `PPL = exp(mean loss)`.

| Measurement | Value |
|---|---:|
| `ln(V)` | {pp['uniform_loss']:.6f} |
| Untrained loss | {pp['untrained_loss']:.6f} |
| Expected uniform PPL | {pp['uniform_perplexity']:,.2f} |
| Observed untrained PPL | {pp['untrained_perplexity']:,.2f} |
| `PPL / V` | {pp['perplexity_to_vocab_ratio']:.6f} |
| Relative distance from `V` | {pp['relative_error']:.2%} |

The untrained perplexity was only **{pp['relative_error']:.2%}** from
the vocabulary size, so the sanity check passed.

### 6. Tied versus untied head

One `V x D` matrix contains **{wt['head_matrix_parameters']:,}**
parameters.

| Configuration | Embedding and head | Entire model |
|---|---:|---:|
| Tied | {wt['tied_embedding_head_unique_parameters']:,} | {wt['tied_total_parameters']:,} |
| Untied | {wt['untied_embedding_head_unique_parameters']:,} | {wt['untied_total_parameters']:,} |

Tying saved:

- **{wt['parameters_saved']:,} parameters**
- **{wt['fp32_mib_saved']:.2f} MiB** of FP32 weights
- Approximately **{wt['estimated_adamw_training_mib_saved']:.2f} MiB**
  of FP32 parameter, gradient, and AdamW state storage

### 7. Materialized versus chunked cross-entropy

The benchmark used `{mem['benchmark_tokens']:,}` hidden states and
chunk size `{mem['chunk_size']:,}`.

| Method | Peak allocation | Loss | Runtime |
|---|---:|---:|---:|
| Materialized | {mem['materialized_peak_mib']:,.2f} MiB | {mem['materialized_loss']:.8f} | {mem['materialized_seconds']:.3f} s |
| Chunked | {mem['chunked_peak_mib']:,.2f} MiB | {mem['chunked_loss']:.8f} | {mem['chunked_seconds']:.3f} s |

The chunked implementation reduced measured peak allocation by
**{mem['measured_reduction_ratio']:.2f}x**. The absolute loss difference
was **{mem['absolute_loss_difference']:.10f}**, and the gradients also
agreed within floating-point tolerance.

## Part 2: Multi-Token Prediction

Head 1 predicts `x[t+1]` and Head 2 predicts `x[t+2]` from the same
hidden state `h[t]`.

The objective is:

    L_total = L_1 + L_2

### Smoke-test losses (not the submitted production values)

| Metric | Value |
|---|---:|
| Head 1, `t+1` | {mtp['head1_final_loss']:.6f} |
| Head 2, `t+2` | {mtp['head2_final_loss']:.6f} |
| Combined loss | {mtp['combined_final_loss']:.6f} |
| Final Head 2 minus Head 1 | {mtp['final_head2_minus_head1']:+.6f} |

### Training change

| Metric | Head 1 | Head 2 |
|---|---:|---:|
| Initial loss | {mtp['head1_initial_loss']:.6f} | {mtp['head2_initial_loss']:.6f} |
| Final loss | {mtp['head1_final_loss']:.6f} | {mtp['head2_final_loss']:.6f} |
| Reduction | {mtp['head1_reduction']:.6f} | {mtp['head2_reduction']:.6f} |
| Active targets | {mtp['head1_target_count']:,} | {mtp['head2_target_count']:,} |

The second head added **{mtp['head2_parameters']:,} parameters**, or
**{mtp['head2_parameters'] * 4 / 2**20:.2f} MiB** in FP32.

### Interpretation

{mtp_analysis}

## Scope

This submission covers the required Part 1 loss harness and Part 2 `t+2`
prediction head.
"""

readme = textwrap.dedent(readme).strip() + "\n"

output_directory = Path.cwd()
readme_path = output_directory / "README.md"
metrics_path = output_directory / "assignment9_metrics.json"

readme_path.write_text(readme, encoding="utf-8")

metrics_path.write_text(
    json.dumps(
        submission_metrics,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

assert readme_path.exists()
assert metrics_path.exists()

saved_metrics = json.loads(
    metrics_path.read_text(encoding="utf-8")
)

assert math.isclose(
    saved_metrics["part2"]["combined_final_loss"],
    mtp["combined_final_loss"],
    rel_tol=0.0,
    abs_tol=1e-12,
)

print("Submission synthesis complete")
print(f"README : {readme_path.resolve()}")
print(f"Metrics: {metrics_path.resolve()}")
print()
print("STEP 9 PASSED")

Submission synthesis complete
README : C:\Users\udisi\Documents\Codex\2026-08-23\one-notebook-one-loss-harness-and\submission\Week 9\README.md
Metrics: C:\Users\udisi\Documents\Codex\2026-08-23\one-notebook-one-loss-harness-and\submission\Week 9\assignment9_metrics.json

STEP 9 PASSED


In [12]:
import gc
import time


# ---------------------------------------------------------------------
# Release the small-model CUDA objects.
# The metrics and generated files remain available on CPU/disk.
# ---------------------------------------------------------------------

for name in [
    "model",
    "output_head",
    "mtp_head2",
    "optimizer",
    "training_parameters",
]:
    globals().pop(name, None)

for name, value in list(globals().items()):
    if torch.is_tensor(value) and value.is_cuda:
        globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()


# ---------------------------------------------------------------------
# Proposed scale configuration
# ---------------------------------------------------------------------

SCALE_V = 8192
SCALE_D = 384
SCALE_T = 512
SCALE_LAYERS = 6
SCALE_HEADS = 6
SCALE_FFN = 1536

SCALE_CHUNK_SIZE = 512
SCALE_LR = 3e-4

MICROBATCH_CANDIDATES = [1, 2, 4, 8]
SAFE_MEMORY_FRACTION = 0.80

assert SCALE_D % SCALE_HEADS == 0
assert SCALE_V == VOCAB_SIZE

torch.manual_seed(9202026)
torch.cuda.manual_seed_all(9202026)


# ---------------------------------------------------------------------
# Select dense 512-token sequences
# ---------------------------------------------------------------------

maximum_required_rows = max(MICROBATCH_CANDIDATES)
preferred_rows = []
fallback_rows = []

with gzip.open(SEQUENCES_PATH, "rt", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)

        if (
            int(row["sequence_length"]) == SCALE_T
            and int(row["loss_bearing_tokens"]) >= 350
        ):
            fallback_rows.append(row)

            if row["lane"] == "general":
                preferred_rows.append(row)

        if (
            len(preferred_rows) >= maximum_required_rows
            and len(fallback_rows) >= maximum_required_rows
        ):
            break

scale_rows = (
    preferred_rows[:maximum_required_rows]
    if len(preferred_rows) >= maximum_required_rows
    else fallback_rows[:maximum_required_rows]
)

assert len(scale_rows) == maximum_required_rows, (
    f"Only found {len(scale_rows)} suitable 512-token sequences"
)

scale_arrays = [
    load_sequence(row)
    for row in scale_rows
]

scale_tokens = torch.as_tensor(
    np.stack([row["input_ids"] for row in scale_arrays]),
    dtype=torch.long,
    device=DEVICE,
)

scale_loss_mask = torch.as_tensor(
    np.stack([row["loss_mask"] for row in scale_arrays]),
    dtype=torch.bool,
    device=DEVICE,
)

scale_segments = torch.as_tensor(
    np.stack([row["segment_ids"] for row in scale_arrays]),
    dtype=torch.long,
    device=DEVICE,
)

scale_positions = torch.as_tensor(
    np.stack([row["position_ids"] for row in scale_arrays]),
    dtype=torch.long,
    device=DEVICE,
)


# ---------------------------------------------------------------------
# Build the larger trunk and MTP head
# ---------------------------------------------------------------------

scale_model = ObservableDecoder(
    vocab_size=SCALE_V,
    maximum_length=SCALE_T,
    hidden_size=SCALE_D,
    num_heads=SCALE_HEADS,
    feed_forward_size=SCALE_FFN,
    num_layers=SCALE_LAYERS,
).to(DEVICE)

# Head 1 uses the tied embedding matrix through F.linear.
# Head 2 owns an independent V x D matrix.
scale_head2 = nn.Linear(
    SCALE_D,
    SCALE_V,
    bias=False,
).to(DEVICE)

nn.init.normal_(
    scale_head2.weight,
    mean=0.0,
    std=0.02,
)

scale_parameters = unique_trainable_parameters(
    scale_model,
    scale_head2,
)

scale_parameter_count = sum(
    parameter.numel()
    for parameter in scale_parameters
)

assert scale_parameter_count == 17_126_400

scale_optimizer = torch.optim.AdamW(
    scale_parameters,
    lr=SCALE_LR,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.01,
)

# Dynamic loss scaling protects small FP16 gradients.
scale_grad_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True,
)


# ---------------------------------------------------------------------
# Chunked head backward
# ---------------------------------------------------------------------

def backward_head_in_chunks(
    active_hidden_graph: torch.Tensor,
    active_targets: torch.Tensor,
    weight: torch.Tensor,
    chunk_size: int,
    grad_scaler: torch.amp.GradScaler,
) -> tuple[torch.Tensor, float]:
    """
    Detach the active hidden states, backpropagate the vocabulary head
    chunk-by-chunk, and return the accumulated hidden-state gradient.

    The returned gradient is still GradScaler-scaled. Passing it into
    the trunk graph preserves the same global scaling.
    """
    hidden_leaf = (
        active_hidden_graph
        .detach()
        .requires_grad_(True)
    )

    target_count = int(active_targets.numel())
    detached_loss_sum = 0.0

    for start in range(0, target_count, chunk_size):
        end = min(start + chunk_size, target_count)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            chunk_logits = F.linear(
                hidden_leaf[start:end],
                weight,
            )

            chunk_loss_sum = F.cross_entropy(
                chunk_logits,
                active_targets[start:end],
                reduction="sum",
            )

        # Each head is independently normalized by its global count.
        grad_scaler.scale(
            chunk_loss_sum / target_count
        ).backward()

        detached_loss_sum += (
            chunk_loss_sum.detach().float().item()
        )

    assert hidden_leaf.grad is not None

    return (
        hidden_leaf.grad,
        detached_loss_sum / target_count,
    )


# ---------------------------------------------------------------------
# One full MTP optimizer step
# ---------------------------------------------------------------------

def scale_training_step(
    batch_size: int,
) -> dict[str, float | int]:
    scale_model.train()
    scale_head2.train()

    batch_tokens = scale_tokens[:batch_size]
    batch_mask = scale_loss_mask[:batch_size]
    batch_segments = scale_segments[:batch_size]
    batch_positions = scale_positions[:batch_size]

    scale_optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):
        embeddings = (
            scale_model.token_embedding(batch_tokens)
            + scale_model.position_embedding(batch_positions)
        )

        hidden = scale_model.forward_from_embeddings(
            embeddings,
            batch_segments,
        )

        head1_mask, head2_mask = build_mtp_masks(
            batch_mask,
            batch_segments,
        )

        head1_hidden_graph = hidden[:, :-1, :][
            head1_mask
        ]

        head2_hidden_graph = hidden[:, :-2, :][
            head2_mask
        ]

        head1_targets = batch_tokens[:, 1:][
            head1_mask
        ]

        head2_targets = batch_tokens[:, 2:][
            head2_mask
        ]

    head1_hidden_gradient, head1_loss = (
        backward_head_in_chunks(
            head1_hidden_graph,
            head1_targets,
            scale_model.token_embedding.weight,
            SCALE_CHUNK_SIZE,
            scale_grad_scaler,
        )
    )

    head2_hidden_gradient, head2_loss = (
        backward_head_in_chunks(
            head2_hidden_graph,
            head2_targets,
            scale_head2.weight,
            SCALE_CHUNK_SIZE,
            scale_grad_scaler,
        )
    )

    # Send both scaled head gradients through their shared trunk graph
    # in one backward traversal.
    torch.autograd.backward(
        tensors=[
            head1_hidden_graph,
            head2_hidden_graph,
        ],
        grad_tensors=[
            head1_hidden_gradient,
            head2_hidden_gradient,
        ],
    )

    scale_grad_scaler.unscale_(scale_optimizer)

    gradient_norm = torch.nn.utils.clip_grad_norm_(
        scale_parameters,
        max_norm=1.0,
    )

    scale_grad_scaler.step(scale_optimizer)
    scale_grad_scaler.update()

    return {
        "head1_loss": head1_loss,
        "head2_loss": head2_loss,
        "total_loss": head1_loss + head2_loss,
        "head1_targets": int(head1_targets.numel()),
        "head2_targets": int(head2_targets.numel()),
        "gradient_norm": float(gradient_norm),
    }


# ---------------------------------------------------------------------
# Initialize CUDA kernels and AdamW state outside the measurements
# ---------------------------------------------------------------------

print("Warming up the scale model...")

_ = scale_training_step(batch_size=1)
torch.cuda.synchronize()

gc.collect()
torch.cuda.empty_cache()


# ---------------------------------------------------------------------
# Calibrate each microbatch candidate
# ---------------------------------------------------------------------

gpu_total_bytes = torch.cuda.get_device_properties(
    DEVICE
).total_memory

calibration_results = []

for candidate_batch_size in MICROBATCH_CANDIDATES:
    try:
        # Candidate-specific warmup.
        _ = scale_training_step(
            batch_size=candidate_batch_size
        )
        torch.cuda.synchronize()

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

        baseline_bytes = torch.cuda.memory_allocated(
            DEVICE
        )

        torch.cuda.reset_peak_memory_stats(DEVICE)

        start_time = time.perf_counter()

        step_result = scale_training_step(
            batch_size=candidate_batch_size
        )

        torch.cuda.synchronize()

        elapsed_seconds = (
            time.perf_counter() - start_time
        )

        absolute_peak_bytes = (
            torch.cuda.max_memory_allocated(DEVICE)
        )

        incremental_peak_bytes = (
            absolute_peak_bytes - baseline_bytes
        )

        physical_tokens = (
            candidate_batch_size * SCALE_T
        )

        physical_tokens_per_second = (
            physical_tokens / elapsed_seconds
        )

        useful_head1_tokens_per_second = (
            step_result["head1_targets"]
            / elapsed_seconds
        )

        peak_fraction = (
            absolute_peak_bytes / gpu_total_bytes
        )

        safe = (
            peak_fraction
            <= SAFE_MEMORY_FRACTION
        )

        calibration_results.append({
            "batch_size": candidate_batch_size,
            "status": "PASS",
            "absolute_peak_mib": to_mib(
                absolute_peak_bytes
            ),
            "incremental_peak_mib": to_mib(
                incremental_peak_bytes
            ),
            "peak_fraction": peak_fraction,
            "seconds": elapsed_seconds,
            "physical_tokens_per_second": (
                physical_tokens_per_second
            ),
            "useful_tokens_per_second": (
                useful_head1_tokens_per_second
            ),
            "head1_loss": step_result["head1_loss"],
            "head2_loss": step_result["head2_loss"],
            "safe": safe,
        })

    except torch.OutOfMemoryError:
        scale_optimizer.zero_grad(set_to_none=True)

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

        calibration_results.append({
            "batch_size": candidate_batch_size,
            "status": "OOM",
            "safe": False,
        })


# ---------------------------------------------------------------------
# Select the largest candidate below 80% of total VRAM
# ---------------------------------------------------------------------

safe_results = [
    row
    for row in calibration_results
    if row["status"] == "PASS"
    and row["safe"]
]

assert safe_results, (
    "No candidate remained below the safe memory threshold."
)

recommended_result = max(
    safe_results,
    key=lambda row: row["batch_size"],
)

recommended_microbatch = (
    recommended_result["batch_size"]
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print()
print("Scale model")
print(
    f"  Configuration : "
    f"D={SCALE_D}, L={SCALE_LAYERS}, "
    f"H={SCALE_HEADS}, FFN={SCALE_FFN}"
)
print(
    f"  Parameters    : "
    f"{scale_parameter_count:,}"
)
print(
    f"  FP32 weights  : "
    f"{to_mib(scale_parameter_count * 4):.2f} MiB"
)
print(
    f"  GPU capacity  : "
    f"{to_mib(gpu_total_bytes):,.2f} MiB"
)

print()
print("Microbatch calibration")
print(
    f"{'B':>3} | "
    f"{'status':<6} | "
    f"{'peak MiB':>10} | "
    f"{'VRAM %':>7} | "
    f"{'step sec':>9} | "
    f"{'physical tok/s':>15} | "
    f"{'useful tok/s':>13}"
)
print("-" * 82)

for row in calibration_results:
    if row["status"] == "OOM":
        print(
            f"{row['batch_size']:>3} | "
            f"{'OOM':<6}"
        )
        continue

    print(
        f"{row['batch_size']:>3} | "
        f"{row['status']:<6} | "
        f"{row['absolute_peak_mib']:>10.2f} | "
        f"{row['peak_fraction']:>6.1%} | "
        f"{row['seconds']:>9.3f} | "
        f"{row['physical_tokens_per_second']:>15,.0f} | "
        f"{row['useful_tokens_per_second']:>13,.0f}"
    )

print()
print(
    f"Recommended microbatch size: "
    f"{recommended_microbatch}"
)
print(
    f"Physical tokens per microbatch: "
    f"{recommended_microbatch * SCALE_T:,}"
)
print(
    "Safety rule: largest tested batch using no more "
    "than 80% of total VRAM."
)

print()
print("SCALE CALIBRATION PASSED")

Warming up the scale model...



Scale model
  Configuration : D=384, L=6, H=6, FFN=1536
  Parameters    : 17,126,400
  FP32 weights  : 65.33 MiB
  GPU capacity  : 8,191.50 MiB

Microbatch calibration
  B | status |   peak MiB |  VRAM % |  step sec |  physical tok/s |  useful tok/s
----------------------------------------------------------------------------------
  1 | PASS   |     404.81 |   4.9% |     0.030 |          17,341 |        17,307
  2 | PASS   |     520.03 |   6.3% |     0.044 |          23,254 |        23,209
  4 | PASS   |     725.19 |   8.9% |     0.054 |          37,939 |        37,865
  8 | PASS   |    1140.15 |  13.9% |     0.081 |          50,775 |        50,676

Recommended microbatch size: 8
Physical tokens per microbatch: 4,096
Safety rule: largest tested batch using no more than 80% of total VRAM.

SCALE CALIBRATION PASSED


In [13]:
# ---------------------------------------------------------------------
# Retained V2 production trajectory (analysis only; does not train)
# ---------------------------------------------------------------------

production_ledger_path = Path(
    "outputs/mtp_17m_session_run/metrics.jsonl"
)
production_summary_path = Path(
    "outputs/mtp_17m_session_run/run_summary.json"
)

production_rows = [
    json.loads(line)
    for line in production_ledger_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
production_summary = json.loads(
    production_summary_path.read_text(encoding="utf-8")
)

# Keep the initial check and the 15 scheduled held-out checks.
# Final audit rows duplicate represented checkpoints and are excluded.
production_validations = [
    row for row in production_rows
    if row.get("event") == "validation"
    and row.get("scope") in {"initial", "periodic"}
]
periodic_validations = [
    row for row in production_validations
    if row.get("scope") == "periodic"
]

assert len(production_validations) == 16
assert len(periodic_validations) == 15
best_ledger_row = min(
    production_validations,
    key=lambda row: float(row["total_loss"]),
)
assert int(best_ledger_row["global_step"]) == int(
    production_summary["best_validation_step"]
)
assert all(
    float(row["head2_loss"]) > float(row["head1_loss"])
    for row in periodic_validations
)

print("RETAINED V2 PRODUCTION VALIDATION TRAJECTORY")
print(" step |      H1 t+1 |      H2 t+2 |         sum | best?")
print("-" * 58)
for row in production_validations:
    best_label = "yes" if row["is_best"] else "no"
    print(
        f"{int(row['global_step']):>5} | "
        f"{float(row['head1_loss']):>11.6f} | "
        f"{float(row['head2_loss']):>11.6f} | "
        f"{float(row['total_loss']):>11.6f} | {best_label}"
    )

best_production = production_summary["best_validation"]
final_production = production_summary["final_validation"]
print()
print("PRODUCTION RUN INTERPRETATION")
print(f"  Best retained step    : {int(production_summary['best_validation_step']):,}")
print(f"  Early-stopping step    : {int(production_summary['global_step']):,}")
print(f"  Best held-out H1       : {best_production['head1_loss']:.6f}")
print(f"  Best held-out H2       : {best_production['head2_loss']:.6f}")
print(f"  Best held-out sum      : {best_production['total_loss']:.6f}")
print(f"  Stopping held-out sum  : {final_production['total_loss']:.6f}")
print("  H2 > H1 checks         : 15 / 15 periodic validations")
print(f"  Stop reason            : {production_summary['stop_reason']}")
print("  Result source          : retained JSONL ledger and run summary; no training launched")
print("PRODUCTION LEDGER CHECK PASSED")


RETAINED V2 PRODUCTION VALIDATION TRAJECTORY
 step |      H1 t+1 |      H2 t+2 |         sum | best?
----------------------------------------------------------
    0 |    9.042049 |    9.211906 |   18.253955 | yes
  500 |    4.074287 |    4.390004 |    8.464290 | yes
 1000 |    3.685883 |    4.121349 |    7.807232 | yes
 1500 |    3.470947 |    3.965115 |    7.436063 | yes
 2000 |    3.274459 |    3.831012 |    7.105471 | yes
 2500 |    3.213202 |    3.789976 |    7.003178 | yes
 3000 |    3.095696 |    3.709081 |    6.804777 | yes
 3500 |    3.076555 |    3.703057 |    6.779611 | yes
 4000 |    2.960565 |    3.619298 |    6.579862 | yes
 4500 |    2.669955 |    3.417056 |    6.087010 | yes
 5000 |    2.928410 |    3.592104 |    6.520515 | no
 5500 |    2.612559 |    3.375047 |    5.987606 | yes
 6000 |    2.893832 |    3.580506 |    6.474338 | no
 6500 |    2.595900 |    3.372842 |    5.968742 | yes
 7000 |    2.839236 |    3.540385 |    6.379621 | no
 7500 |    2.635009 |    3.394982

In [14]:
# ---------------------------------------------------------------------
# Final submission values from the retained production checkpoint
# ---------------------------------------------------------------------

import subprocess
import sys

production_run_dir = Path(
    "outputs/mtp_17m_session_run"
)
production_summary_path = (
    production_run_dir / "run_summary.json"
)

assert production_summary_path.exists(), (
    "Run the production MTP training section before final synthesis: "
    f"missing {production_summary_path}"
)

subprocess.run(
    [
        sys.executable,
        "outputs/build_assignment_submission.py",
        "--run-dir",
        str(production_run_dir),
    ],
    check=True,
)

final_metrics = json.loads(
    Path("assignment9_final_metrics.json")
    .read_text(encoding="utf-8")
)
production = final_metrics["part2_production"]
best = production["best_validation"]

print()
print("PART 2 PRODUCTION RESULTS — SUBMITTED CHECKPOINT")
print("  Earlier 80-update values are smoke-test evidence only.")
print("  Part 1 values: assignment9_final_metrics.json -> part1")
print(f"  Best production step : {production['best_step']:,}")
print(f"  Early-stopping step   : {production['stopping_step']:,}")
print(f"  Head 1 held-out loss : {best['head1_loss']:.6f}")
print(f"  Head 2 held-out loss : {best['head2_loss']:.6f}")
print(f"  Combined held-out    : {best['total_loss']:.6f}")
print(f"  Stop reason          : {production['stop_reason']}")
print(f"  Best checkpoint      : {production['best_checkpoint_path']}")
print("  Complete write-up     : ASSIGNMENT_WRITEUP.md")
print("FINAL SYNTHESIS PASSED")


PART 2 PRODUCTION RESULTS — SUBMITTED CHECKPOINT
  Earlier 80-update values are smoke-test evidence only.
  Part 1 values: assignment9_final_metrics.json -> part1
  Best production step : 6,500
  Early-stopping step   : 7,500
  Head 1 held-out loss : 2.595900
  Head 2 held-out loss : 3.372842
  Combined held-out    : 5.968742
  Stop reason          : validation_early_stopping
  Best checkpoint      : outputs\mtp_17m_session_run\best.pt
  Complete write-up     : ASSIGNMENT_WRITEUP.md
FINAL SYNTHESIS PASSED
